# Catalog Plot Testing

Exploratory and non-paper plot cells moved out of notebook 8. These cells are kept for interactive testing and may depend on earlier helper cells in this notebook.

In [ ]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_candidates = (_here, *_here.parents)
REPO_ROOT = next((candidate for candidate in _candidates if (candidate / "m33_pipeline").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook

REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f"Notebook working directory set to: {REPO_ROOT}")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
import astropy.constants as c
import pyneb as pn
import matplotlib.pyplot as plt
HAS_PYNEB=True
from astropy.coordinates import SkyCoord, SkyOffsetFrame
from astropy.visualization import AsinhStretch, PercentileInterval
from skimage.segmentation import find_boundaries
from matplotlib import cm
from skimage.measure import find_contours
from matplotlib.patches import ConnectionPatch
from scipy.optimize import curve_fit
from types import SimpleNamespace
import os
import shutil
import re
import matplotlib as mpl
import seaborn as sns

from m33_pipeline.reporting import (
    add_fit_values,
    add_duplicate_region_values,
    add_halpha_flux_partition_values,
    build_catalog_number_values,
    format_value,
    load_snr_catalog,
    load_wr_catalog,
    write_latex_commands,
)
from m33_pipeline.planetary_nebulae import crossmatch_pne_to_hii_boundaries

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "axes.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

plt.rcParams["text.usetex"] = False

plt.rc('text', usetex=False)
plt.rc('font', family='serif', size=20)

plt.rc('xtick', direction='in', top=True)
plt.rc('ytick', direction='in', right=True)
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

from m33_pipeline import paths


In [ ]:
# Toggle this to also copy selected final figures into PAPER_PLOTS_DRAFT1.
SAVE_FOR_PAPER = True
PAPER_DRAFT_DIR = Path(globals().get('REPO_ROOT', globals().get('ROOT', Path.cwd()))) / 'PAPER_PLOTS_DRAFT1'
PAPER_FIGURE_EXPORTS = {'M33_HII_radial_gradients_multipanel_with_histograms.png': 'M33_radial_gradients_histogram.png', 'M33_allfields_BPT_SNR3_SF_black_gray_compact_objects.png': 'M33_BPT.png', 'M33_HII_Halpha_luminosity_function_correction_comparison_empirical_pdf.png': 'M33_LF_slopes.png', 'M33_HII_L_Ha_sum_dered_vs_areaeq.png': 'M33_L_R.png', 'M33_metallicity_calibrations_paper_multipanel.png': 'M33_metallicity_gradients_selected.png', 'M33_metallicity_vs_logU_pressure_grid.png': 'M33_metallicity_vs_logU_pressure_grid.png', 'M33_metallicity_calibrations_all_indicators_multipanel.png': 'M33_metallicity_calibrations_all_indicators_multipanel.png'}

def save_for_paper(source_path, draft_filename=None):
    if not SAVE_FOR_PAPER:
        return None
    source_path = Path(source_path)
    target_name = draft_filename or PAPER_FIGURE_EXPORTS.get(source_path.name)
    if target_name is None:
        return None
    import shutil
    PAPER_DRAFT_DIR.mkdir(parents=True, exist_ok=True)
    target_path = PAPER_DRAFT_DIR / target_name
    shutil.copyfile(source_path, target_path)
    print(f'Saved paper draft figure: {target_path}')
    return target_path


In [ ]:
#make a list of 15 evenly spaced colours from the rainbow cmap to be used in all plots
cmap = plt.get_cmap('rainbow')
colors = [cmap(i) for i in np.linspace(0, 1, 15)]
#show the colours
plt.figure(figsize=(8, 2))
for i, color in enumerate(colors):
    plt.plot([i, i + 1], [0, 0], color=color, lw=4)
#label each colour with its index
for i in range(len(colors)):
    plt.text(i + 0.5, 0.1, str(i), ha='center', va='bottom')
plt.xlim(0, len(colors) + 1)
plt.ylim(-1, 1)
plt.axis('off')
plt.title('Rainbow Colormap')
plt.tight_layout()

# Shared plot colors. Non-metallicity catalog-property plots use the green-to-red
# part of rainbow; metallicity plots use a separate blue-purple rainbow slice.
property_cmap = plt.get_cmap('rainbow')
property_color_positions = {
    'sum_A_V': 0.38,
    'L_Ha_sum_dered': 0.52,
    'ne_SII_cm3': 0.66,
    'logU_KK04': 0.76,
    'log_P_thermal_SII_over_k': 0.86,
    'radius_areaeq_pc': 0.94,
}
color_dic = {key: property_cmap(pos) for key, pos in property_color_positions.items()}
lf_comparison_colors = {
    'pre_dig': property_cmap(0.48),
    'dig_corrected': color_dic['L_Ha_sum_dered'],
}



In [ ]:
# load in the latest HII region catalog
catalog_method = 'summed_map'  # options: integrated_spectrum, summed_map
catalog_dig_mode = 'dig_subtracted'  # options: dig_subtracted, no_dig
catalog_variant = 'combined'  # options: combined, metallicity, ionization_parameter, density, other_derived, clustering
primary_only = True
field_catalog_pattern = f"flux_catalogs/{catalog_method}/{catalog_dig_mode}/flux_catalog_{{FIELD}}.csv"

plot_root = Path("PAPER_PLOTS") / catalog_method / catalog_dig_mode
plot_root.mkdir(parents=True, exist_ok=True)

def paper_plot_path(filename, subdir=None):
    outdir = plot_root if subdir is None else plot_root / subdir
    outdir.mkdir(parents=True, exist_ok=True)
    return outdir / filename

def resolve_catalog_path(method, dig_mode, variant):
    base_dir = Path("CATALOGS") / "flux_catalogs" / method / dig_mode
    if variant == "combined":
        if hasattr(paths, "combined_catalog_csv"):
            return paths.combined_catalog_csv(method=method, dig_mode=dig_mode)
        return base_dir / "total_flux_catalog_combined.csv"
    if variant == "clustering":
        if hasattr(paths, "derived_catalog_csv"):
            return paths.derived_catalog_csv("clustering", method=method, dig_mode=dig_mode)
        return base_dir / "derived" / "total_flux_catalog_clustering.csv"
    if hasattr(paths, "derived_catalog_csv"):
        return paths.derived_catalog_csv(variant, method=method, dig_mode=dig_mode)
    return base_dir / "derived" / f"total_flux_catalog_{variant}.csv"

def load_catalog_with_field_fallback(method, dig_mode, variant):
    """Load the requested total catalog, or concatenate the available field catalogs."""
    cat_path = resolve_catalog_path(method, dig_mode, variant)
    if cat_path.exists():
        return pd.read_csv(cat_path), str(cat_path)

    base_dir = Path("CATALOGS") / "flux_catalogs" / method / dig_mode
    field_paths = sorted(base_dir.glob("flux_catalog_*.csv"))
    if not field_paths:
        raise FileNotFoundError(
            f"Could not find {cat_path} or any field catalogs in {base_dir}"
        )

    field_catalogs = []
    for field_path in field_paths:
        field_cat = pd.read_csv(field_path)
        if "field" not in field_cat.columns:
            field_cat["field"] = field_path.stem.removeprefix("flux_catalog_")
        field_catalogs.append(field_cat)

    source = f"{len(field_paths)} available field catalog(s) from {base_dir}"
    return pd.concat(field_catalogs, ignore_index=True, sort=False), source

cat, cat_source = load_catalog_with_field_fallback(catalog_method, catalog_dig_mode, catalog_variant)
cat_before_primary_filter = cat.copy()
pn_catalog_path = Path('CATALOGS/planetary_nebulae/M33_PNe_combined_deduplicated.csv')
if pn_catalog_path.exists():
    pn_catalog = pd.read_csv(pn_catalog_path)
    pn_hii_matches, cat = crossmatch_pne_to_hii_boundaries(cat, pn_catalog)
    pn_match_path = Path('CATALOGS/planetary_nebulae/M33_PNe_HII_region_crossmatch.csv')
    pn_match_path.parent.mkdir(parents=True, exist_ok=True)
    pn_hii_matches.to_csv(pn_match_path, index=False)
    pn_derived_path = Path('CATALOGS/flux_catalogs') / catalog_method / catalog_dig_mode / 'derived' / 'total_flux_catalog_pn_crossmatch.csv'
    pn_derived_path.parent.mkdir(parents=True, exist_ok=True)
    cat.to_csv(pn_derived_path, index=False)
    print(f'PN boundary crossmatch: {cat["has_pn_in_boundary"].sum()} HII regions contain at least one PN.')
else:
    pn_catalog = pd.DataFrame()
    print(f'PN catalog not found: {pn_catalog_path}')
if primary_only and 'primary' in cat.columns:
    n_before = len(cat)
    n_non_primary = int((~cat['primary'].fillna(True)).sum())
    cat = cat.loc[cat['primary'].fillna(True)].copy().reset_index(drop=True)
    print(f"Primary-only filtering enabled: removed {n_non_primary} non-primary duplicate rows; {len(cat)} / {n_before} rows kept.")
elif primary_only:
    print("Primary-only filtering requested, but no 'primary' column exists in this catalog.")
print(f"Catalog loaded from {cat_source} with {len(cat.columns)} columns.")
print(f"Plots will be saved under {plot_root}")
for col in cat.columns:
    print(col)


In [ ]:
# Star-forming region definition used throughout this notebook.
# Change this one block, then rerun the notebook to switch the selection.
#
# Examples:
#   STAR_FORMING_DEFINITION = {"mode": "bpt"}
#   STAR_FORMING_DEFINITION = {"mode": "sii_ha", "sii_ha_max": 0.4}
#
# For sii_ha mode, sii_ha_max is the linear ratio
# ([SII]6716 + [SII]6731) / Halpha.
STAR_FORMING_DEFINITION = {
    "mode": "bpt",             # options: "bpt", "sii_ha"
    "sii_ha_max": 0.4,      # used only when mode == "sii_ha"
}
STAR_FORMING_SNR_CUT = 3.0


def _first_present_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _numeric_column(df, candidates):
    col = _first_present_column(df, candidates)
    if col is None:
        return np.full(len(df), np.nan, dtype=float), None
    return pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float), col


def _selected_sii_ha_ratio(df):
    ha, ha_col = _numeric_column(df, [
        "F_Halpha_sum_dered",
        "F_Halpha_sum_digsub",
        "F_Halpha_sum",
        "F_Halpha_sum_nodig",
    ])
    sii6716, sii6716_col = _numeric_column(df, [
        "F_[SII]6716_sum_dered",
        "F_[SII]6716_sum_digsub",
        "F_[SII]6716_sum",
        "F_[SII]6716_sum_nodig",
    ])
    sii6731, sii6731_col = _numeric_column(df, [
        "F_[SII]6731_sum_dered",
        "F_[SII]6731_sum_digsub",
        "F_[SII]6731_sum",
        "F_[SII]6731_sum_nodig",
    ])
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = (sii6716 + sii6731) / ha
    ratio = np.where(np.isfinite(ratio) & (ha > 0) & (sii6716 > 0) & (sii6731 > 0), ratio, np.nan)
    return ratio, {"Halpha": ha_col, "SII6716": sii6716_col, "SII6731": sii6731_col}


def selected_star_forming_mask(df, definition=None):
    definition = dict(STAR_FORMING_DEFINITION if definition is None else definition)
    mode = str(definition.get("mode", "bpt")).strip().lower()
    if mode == "bpt":
        if "BPT_class_sum_dered" not in df.columns:
            return np.zeros(len(df), dtype=bool)
        return df["BPT_class_sum_dered"].astype(str).str.strip().eq("Star-forming").to_numpy(dtype=bool)
    if mode == "sii_ha":
        ratio, _ = _selected_sii_ha_ratio(df)
        max_ratio = float(definition.get("sii_ha_max", 0.4))
        return np.isfinite(ratio) & (ratio < max_ratio)
    raise ValueError(f"Unknown STAR_FORMING_DEFINITION mode: {mode!r}")


def selected_star_forming_snr_mask(df, snr_cut=None, definition=None):
    snr_cut = STAR_FORMING_SNR_CUT if snr_cut is None else snr_cut
    if "all_lines_snr_mask" in globals():
        snr_mask = all_lines_snr_mask(df, snr_cut=snr_cut)
    elif "_all_bpt_lines_snr_mask_for_plotting" in globals():
        snr_mask = _all_bpt_lines_snr_mask_for_plotting(df, snr_cut=snr_cut)
    else:
        snr_cols = [
            "SNR_Halpha_sum",
            "SNR_Hbeta_sum",
            "SNR_[OIII]5007_sum",
            "SNR_[SII]6716_sum",
            "SNR_[NII]6583_sum",
        ]
        snr_mask = np.ones(len(df), dtype=bool)
        for col in snr_cols:
            if col not in df.columns:
                snr_mask &= False
            else:
                snr_mask &= pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float) > snr_cut
    primary = df["primary"].fillna(True).astype(bool).to_numpy() if "primary" in df.columns else np.ones(len(df), dtype=bool)
    return selected_star_forming_mask(df, definition=definition) & snr_mask & primary


def star_forming_definition_label(definition=None):
    definition = dict(STAR_FORMING_DEFINITION if definition is None else definition)
    mode = str(definition.get("mode", "bpt")).strip().lower()
    if mode == "bpt":
        return "BPT star-forming"
    if mode == "sii_ha":
        return rf"[SII]/H$\alpha$ < {float(definition.get('sii_ha_max', 0.4)):.2f}"
    return mode

cat["selected_star_forming"] = selected_star_forming_mask(cat)
cat["selected_star_forming_snr"] = selected_star_forming_snr_mask(cat, snr_cut=STAR_FORMING_SNR_CUT)
print(f"Star-forming definition: {star_forming_definition_label()}")
print(f"Selected star-forming regions: {int(cat['selected_star_forming'].sum())} / {len(cat)}")
print(f"Selected star-forming regions with all-line S/N > {STAR_FORMING_SNR_CUT:g}: {int(cat['selected_star_forming_snr'].sum())} / {len(cat)}")



In [ ]:
# Derived BPT diagnostic columns for testing alternate star-forming definitions.
# The high-OIII population is defined as primary, high-S/N, above the Kauffmann+03 line,
# and not currently BPT-classified as star-forming.
BPT_DIAGNOSTIC_SNR_CUT = 3.0
BPT_DIAGNOSTIC_SNR_LINES = [
    'Halpha',
    'Hbeta',
    '[OIII]5007',
    '[NII]6583',
    '[SII]6716',
    '[SII]6731',
    '[OII]3727',
]


def _diag_first_present_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _diag_numeric_column(df, candidates):
    col = _diag_first_present_column(df, candidates)
    if col is None:
        return np.full(len(df), np.nan, dtype=float), None
    return pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float), col


def _diag_flux(df, line_name):
    return _diag_numeric_column(df, [
        f'F_{line_name}_sum_dered',
        f'F_{line_name}_sum_digsub',
        f'F_{line_name}_sum',
        f'F_{line_name}_sum_nodig',
    ])


def _diag_snr(df, line_name):
    return _diag_numeric_column(df, [
        f'SNR_{line_name}_sum',
        f'SNR_{line_name}_sum_nodig',
    ])


def _diag_log_ratio(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        out = np.log10(num / den)
    return np.where(np.isfinite(out) & (num > 0) & (den > 0), out, np.nan)


def _diag_all_line_snr_mask(df, threshold, line_names=BPT_DIAGNOSTIC_SNR_LINES):
    masks = []
    snr_sources = {}
    for line_name in line_names:
        snr_values, snr_col = _diag_snr(df, line_name)
        snr_sources[line_name] = snr_col
        masks.append(np.isfinite(snr_values) & (snr_values > threshold))
    if not masks:
        return np.zeros(len(df), dtype=bool), snr_sources
    return np.logical_and.reduce(masks), snr_sources


def _category_counts(series):
    return series.value_counts(dropna=False).to_dict()


def add_bpt_diagnostic_columns(df, snr_cut=BPT_DIAGNOSTIC_SNR_CUT):
    out = df.copy()
    ha, _ = _diag_flux(out, 'Halpha')
    hb, _ = _diag_flux(out, 'Hbeta')
    oiii, _ = _diag_flux(out, '[OIII]5007')
    oii, _ = _diag_flux(out, '[OII]3727')
    nii, _ = _diag_flux(out, '[NII]6583')
    sii6716, _ = _diag_flux(out, '[SII]6716')
    sii6731, _ = _diag_flux(out, '[SII]6731')
    sii = sii6716 + sii6731

    out['sii_halpha_ratio'] = np.where(np.isfinite(sii / ha) & (sii > 0) & (ha > 0), sii / ha, np.nan)
    out['log_SII_Halpha'] = _diag_log_ratio(sii, ha)
    out['log_NII_Halpha'] = _diag_log_ratio(nii, ha)
    out['log_OIII_OII'] = _diag_log_ratio(oiii, oii)
    out['log_OIII_Halpha'] = _diag_log_ratio(oiii, ha)
    out['log_OIII_Hbeta'] = _diag_log_ratio(oiii, hb)

    with np.errstate(divide='ignore', invalid='ignore'):
        kauffmann_y = 0.61 / (out['log_NII_Halpha'].to_numpy(dtype=float) - 0.05) + 1.3
    x_nii = out['log_NII_Halpha'].to_numpy(dtype=float)
    y_oiii = out['log_OIII_Hbeta'].to_numpy(dtype=float)
    above_kauffmann = np.isfinite(x_nii) & np.isfinite(y_oiii) & ((x_nii >= 0.05) | (y_oiii > kauffmann_y))
    out['above_kauffmann_nii_bpt'] = above_kauffmann

    snr3_all, snr_sources = _diag_all_line_snr_mask(out, snr_cut)
    snr30_all, _ = _diag_all_line_snr_mask(out, 30.0)
    snr50_all, _ = _diag_all_line_snr_mask(out, 50.0)
    out['snr3_all_lines'] = snr3_all
    out['snr30_all_lines'] = snr30_all
    out['snr50_all_lines'] = snr50_all
    out['high_snr_nii_bpt'] = snr3_all

    primary = out['primary'].fillna(True).astype(bool).to_numpy() if 'primary' in out.columns else np.ones(len(out), dtype=bool)
    bpt_sf = out['BPT_class_sum_dered'].astype(str).str.strip().eq('Star-forming').to_numpy() if 'BPT_class_sum_dered' in out.columns else np.zeros(len(out), dtype=bool)
    out['bpt_sf_snr3_all_lines'] = primary & snr3_all & bpt_sf
    out['snr3_non_bpt_sf'] = primary & snr3_all & ~bpt_sf
    out['snr50_non_bpt_sf'] = primary & snr50_all & ~bpt_sf
    out['high_snr_above_kauffmann_non_sf'] = primary & snr3_all & above_kauffmann & ~bpt_sf

    recovered_siiha = primary & snr30_all & ~bpt_sf & np.isfinite(out['sii_halpha_ratio']) & (out['sii_halpha_ratio'] < 0.6)
    out['custom_sf_bpt_or_siiha'] = out['bpt_sf_snr3_all_lines'] | recovered_siiha
    out['custom_sf_recovered_siiha'] = recovered_siiha

    out['class_snr3_only'] = np.where(primary & snr3_all, 'snr3_all_lines', 'low_snr')
    out['class_bpt_snr3_nonbpt'] = np.select(
        [out['bpt_sf_snr3_all_lines'], out['snr3_non_bpt_sf']],
        ['bpt_sf_snr3', 'snr3_non_bpt_sf'],
        default='low_snr',
    )
    out['class_bpt_snr3_nonbpt_snr50'] = np.select(
        [out['bpt_sf_snr3_all_lines'], out['snr50_non_bpt_sf'], out['snr3_non_bpt_sf']],
        ['bpt_sf_snr3', 'snr50_non_bpt_sf', 'snr3_non_bpt_sf'],
        default='low_snr',
    )
    out['class_custom_sf'] = np.select(
        [out['bpt_sf_snr3_all_lines'], out['custom_sf_recovered_siiha'], primary & snr3_all],
        ['bpt_sf_snr3', 'recovered_siiha_sf', 'snr3_non_custom_sf'],
        default='low_snr',
    )

    out.attrs['bpt_diagnostic_snr_sources'] = snr_sources
    return out


cat = add_bpt_diagnostic_columns(cat, snr_cut=BPT_DIAGNOSTIC_SNR_CUT)
print(f"S/N>3 all-line regions: {int(cat['snr3_all_lines'].sum())}")
print(f"BPT SF with S/N>3 all lines: {int(cat['bpt_sf_snr3_all_lines'].sum())}")
print(f"S/N>3 all lines but not BPT SF: {int(cat['snr3_non_bpt_sf'].sum())}")
print(f"S/N>50 all lines and not BPT SF: {int(cat['snr50_non_bpt_sf'].sum())}")
print(f"Recovered SII/Halpha SF candidates: {int(cat['custom_sf_recovered_siiha'].sum())}")
print(f"High-S/N, above-Kauffmann, non-SF regions: {int(cat['high_snr_above_kauffmann_non_sf'].sum())}")
print('S/N columns used:', cat.attrs.get('bpt_diagnostic_snr_sources', {}))



In [ ]:
# load in the WR and SNR catalogs
wr_catalog = load_wr_catalog()
snr_catalog = load_snr_catalog()


## Cells From 8a Not Used By Paper Notebook Or BPT Comparison

In [ ]:
# save the catalog to a latex table

In [ ]:
# Initialize values that will be extended with fit results later in the notebook.
values, formats = build_catalog_number_values(cat, snr_cut=3)
add_duplicate_region_values(values, cat_before_primary_filter)
if {'has_snr_in_boundary', 'has_wr_in_boundary'}.issubset(cat.columns):
    snr_hosts = cat['has_snr_in_boundary'].fillna(False).astype(bool)
    wr_hosts = cat['has_wr_in_boundary'].fillna(False).astype(bool)
    values['nRegionsContainingSNR'] = int(snr_hosts.sum())
    values['nRegionsContainingWR'] = int(wr_hosts.sum())
    values['nRegionsContainingSNRAndWR'] = int((snr_hosts & wr_hosts).sum())
    values['nRegionsContainingSNROrWR'] = int((snr_hosts | wr_hosts).sum())
if 'n_pn_in_boundary' in cat.columns:
    values['nCrossMatchedPN'] = int(pd.to_numeric(cat['n_pn_in_boundary'], errors='coerce').fillna(0).sum())
radial_gradient_fit_results = {}

print("\nCatalog and peak-removal commands initialized:")
for k in values:
    print(f"\\{k} = {format_value(values[k], formats.get(k))}")


In [ ]:
# plot each of the 9 fields separately, with the same settings and save them

plot_all_m33_fields_separately(
    field_order=("NE", "NW", "F5", "SE", "SW", "F7", "F6", "F8", "F9"),
    output_dir=str(paper_plot_path("individual_fields", subdir="field_maps")),
    ha_dir="../M33-Maps",
    ha_filename_pattern="M33-{FIELD}/M33-{FIELD}_SN3.LineMaps.map.Ha+OIII.1x1.amplitude.fits",
    catalog_dir="CATALOGS",
    catalog_pattern=field_catalog_pattern,
    boundary_dir="Boundary_maps/Boundary_map_100pc",
    boundary_pattern="Boundary_map_{FIELD}.fits",   # use the true label map
    wcs_pattern="M33-{FIELD}/M33{FIELD}-Haflux.fits",
    xlim=(50, 2000),
    ylim=(50, 2000),
    cmap=cm.get_cmap("rainbow").copy(),
    vmin=-18,
    vmax=-15.0,
    region_id_col="region_id",
    bpt_col="BPT_class_sum_dered",
    class_colors={
        "Star-forming": "navy",
        "Composite": "darkgreen",
        "AGN/Shock": "maroon",
        "Unclassified": "gray",
    },
    boundary_lw=0.5,
    boundary_alpha=0.9,
    plot_pn=True,
    pn_color="royalblue",
    pn_catalog=pn_catalog,
    figsize=(9, 9),
    filetype="pdf",
    
)


# Separate grayscale field overlays highlighting regions that contain SNRs or WR stars.
plot_all_m33_fields_separately(
    field_order=("NE", "NW", "F5", "SE", "SW", "F7", "F6", "F8", "F9"),
    output_dir=str(paper_plot_path("individual_fields_wr_snr", subdir="field_maps")),
    ha_dir="../M33-Maps",
    ha_filename_pattern="M33-{FIELD}/M33-{FIELD}_SN3.LineMaps.map.Ha+OIII.1x1.amplitude.fits",
    catalog_dir="CATALOGS",
    catalog_pattern=field_catalog_pattern,
    boundary_dir="Boundary_maps/Boundary_map_100pc",
    boundary_pattern="Boundary_map_{FIELD}.fits",
    wcs_pattern="M33-{FIELD}/M33{FIELD}-Haflux.fits",
    xlim=(50, 2000),
    ylim=(50, 2000),
    cmap=cm.get_cmap("gray_r").copy(),
    vmin=-18,
    vmax=-15.0,
    region_id_col="region_id",
    boundary_color_mode="wr_snr",
    combined_catalog=cat,
    wr_region_color="darkviolet",
    snr_region_color="darkorange",
    both_region_color="crimson",
    pn_region_color="royalblue",
    other_region_color="navy",
    plot_wr=True,
    plot_snr=True,
    plot_pn=True,
    wr_color="darkviolet",
    snr_color="yellow",
    pn_color="royalblue",
    boundary_lw=0.4,
    boundary_alpha=0.95,
    figsize=(9, 9),
    filename_suffix="WR_SNR_PN_overlay_grayscale",
    pn_catalog=pn_catalog,
    filetype="pdf",
)


In [ ]:
# #plot the NW halpha flux map
# ha_flux_nw = 

In [ ]:
# number of selected star-forming regions
sf_count = int(selected_star_forming_mask(cat).sum())
print(f"Star-forming definition: {star_forming_definition_label()}")
print(f'Number of selected star-forming regions: {sf_count} out of {len(cat)}')

In [ ]:
np.nanmin(cat['SNR_Halpha_sum'])

#how many regions have SNR_Halpha_sum <= 3.0?
low_snr_count = np.sum(cat['SNR_Halpha_sum'] <= 3.0)
print(f"Number of regions with SNR_Halpha_sum <= 3.0: {low_snr_count}")

In [ ]:
# min(cat['area_px_after_carve'])

#how many regions have area_px_after_carve <= 20 and low Halpha flux?
small_region_count = np.sum((cat['area_px_after_carve'] <= 20))
print(f"Number of regions with area_px_after_carve <= 20 and F_Halpha <= 10.0: {small_region_count}")
#which field are these regions in?
small_regions = cat[(cat['area_px_after_carve'] <= 20)]
print(small_regions[['field', 'area_px_after_carve', 'F_Halpha_sum_dered']])

In [ ]:
def load_available_flux_catalog(method, dig_mode, primary_only=True):
    base_dir = Path('CATALOGS') / 'flux_catalogs' / method / dig_mode
    preferred_paths = [
        base_dir / 'total_flux_catalog_combined.csv',
        base_dir / 'total_flux_catalog.csv',
    ]

    df = None
    source = None
    for path in preferred_paths:
        if path.exists():
            df = pd.read_csv(path)
            source = str(path)
            break

    if df is None:
        field_paths = sorted(base_dir.glob('flux_catalog_*.csv'))
        if not field_paths:
            raise FileNotFoundError(f'No available flux catalogs found in {base_dir}')
        df = pd.concat((pd.read_csv(path) for path in field_paths), ignore_index=True, sort=False)
        source = f'{len(field_paths)} field catalogs from {base_dir}'

    if primary_only and 'primary' in df.columns:
        df = df.loc[df['primary'].fillna(True)].copy().reset_index(drop=True)

    return df, source


cat_nodig, nodig_source = load_available_flux_catalog(catalog_method, 'no_dig', primary_only=primary_only)
cat_digsub, digsub_source = load_available_flux_catalog(catalog_method, 'dig_subtracted', primary_only=primary_only)

dig_before = cat_nodig['log_L_Ha_sum_dered'].to_numpy(dtype=float)
dig_after = cat_digsub['log_L_Ha_sum_dered'].to_numpy(dtype=float)

dig_before = dig_before[np.isfinite(dig_before)]
dig_after = dig_after[np.isfinite(dig_after)]

def luminosity_function_pdf_points(logL, bins):
    logL = np.asarray(logL, dtype=float)
    logL = logL[np.isfinite(logL)]
    if len(logL) == 0:
        return np.array([]), np.array([]), np.array([])
    counts, edges = np.histogram(logL, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)
    pdf = counts / (np.sum(counts) * widths)
    detected = counts > 0
    return centers[detected], pdf[detected], counts[detected]

logL_bins = np.arange(33.0, 40.6, 0.2)
fig, ax = plt.subplots(figsize=(8, 6))

lf_specs = [
    (dig_before, lf_comparison_colors['pre_dig'], 'Pre-background/DIG correction'),
    (dig_after, lf_comparison_colors['dig_corrected'], 'DIG corrected'),
]

for logL_values, color, label in lf_specs:
    centers, pdf, counts = luminosity_function_pdf_points(logL_values, logL_bins)
    if len(centers) == 0:
        continue
    ax.plot(
        centers,
        pdf,
        marker='o',
        markersize=6,
        linewidth=2.5,
        color=color,
        label=f'{label} (N={len(logL_values)})',
    )

ax.set_xlabel(r'log$_{10}$[$L_{\rm H\alpha}$ (erg s$^{-1}$)]')
ax.set_ylabel(r'Probability density per dex')
ax.set_yscale('log')
ax.minorticks_on()
ax.legend(frameon=False, fontsize=12)
fig.tight_layout()
fig.savefig(
    paper_plot_path(
        f'M33_HII_Halpha_luminosity_function_corrections_{catalog_method}.png',
        subdir='luminosity_functions'
    ),
    dpi=300,
    bbox_inches='tight'
)
plt.show()

print(f'DIG comparison source, no_dig: {nodig_source}')
print(f'DIG comparison source, dig_subtracted: {digsub_source}')


# Three-version luminosity function: raw observed, extinction corrected, and DIG+extinction corrected.
def _first_log_luminosity(df, candidates):
    for col in candidates:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
            if col.startswith('L_'):
                with np.errstate(divide='ignore', invalid='ignore'):
                    vals = np.log10(vals)
            return vals, col
    raise KeyError(f'None of these luminosity columns are present: {candidates}')


def _fit_lf_pdf_slope(logL_values, bins, min_points=4):
    centers, pdf, counts = luminosity_function_pdf_points(logL_values, bins)
    mask = np.isfinite(centers) & np.isfinite(pdf) & (pdf > 0) & (counts > 0)
    if mask.sum() < min_points:
        return centers, pdf, counts, None
    fit = linregress(centers[mask], np.log10(pdf[mask]))
    return centers, pdf, counts, fit

lf_three_specs = []
raw_logL, raw_col = _first_log_luminosity(cat_nodig, ['log_L_Ha_sum', 'L_Ha_sum', 'log_L_Ha_observed', 'L_Ha_observed'])
ext_logL, ext_col = _first_log_luminosity(cat_nodig, ['log_L_Ha_sum_dered', 'L_Ha_sum_dered'])
final_logL, final_col = _first_log_luminosity(cat_digsub, ['log_L_Ha_sum_dered', 'L_Ha_sum_dered'])
lf_three_specs = [
    (raw_logL, '0.62', 'Raw observed', raw_col, 1),
    (ext_logL, 'black', 'Extinction corrected', ext_col, 2),
    (final_logL, color_dic['L_Ha_sum_dered'], 'DIG + extinction corrected', final_col, 3),
]

fig, ax = plt.subplots(figsize=(8.5, 6.4))
for logL_values, color, label, col, zorder in lf_three_specs:
    logL_values = np.asarray(logL_values, dtype=float)
    logL_values = logL_values[np.isfinite(logL_values)]
    centers, pdf, counts, fit = _fit_lf_pdf_slope(logL_values, logL_bins)
    if len(centers) == 0:
        continue
    fit_label = f'{label} ({col}, N={len(logL_values)})'
    if fit is not None:
        fit_label += rf'; slope={fit.slope:.2f}$\pm${fit.stderr:.2f}'
    ax.plot(centers, pdf, marker='o', markersize=5.5, linewidth=2.2, color=color, label=fit_label, zorder=zorder)
    if fit is not None:
        xfit = np.linspace(np.nanmin(centers), np.nanmax(centers), 200)
        yfit = 10 ** (fit.slope * xfit + fit.intercept)
        ax.plot(xfit, yfit, linestyle='--', linewidth=1.7, color=color, zorder=zorder)

ax.set_xlabel(r'log$_{10}$[$L_{\rm H\alpha}$ (erg s$^{-1}$)]')
ax.set_ylabel(r'Probability density per dex')
ax.set_yscale('log')
ax.minorticks_on()
ax.legend(frameon=False, fontsize=10)
fig.tight_layout()
fig.savefig(
    paper_plot_path('M33_HII_Halpha_luminosity_function_raw_extcorr_digextcorr.png', subdir='luminosity_functions'),
    dpi=300,
    bbox_inches='tight',
)
plt.show()


In [ ]:
# Shared color dictionaries are initialized near the top of the notebook so early plots can use them.
# Recompute here as a harmless refresh before the radial-gradient section.
property_cmap = plt.get_cmap('rainbow')
property_color_positions = {
    'sum_A_V': 0.38,
    'L_Ha_sum_dered': 0.52,
    'ne_SII_cm3': 0.66,
    'logU_KK04': 0.76,
    'log_P_thermal_SII_over_k': 0.86,
    'radius_areaeq_pc': 0.94,
}
color_dic = {key: property_cmap(pos) for key, pos in property_color_positions.items()}
lf_comparison_colors = {
    'pre_dig': property_cmap(0.48),
    'dig_corrected': color_dic['L_Ha_sum_dered'],
}


In [ ]:
for col in cat.columns:
    print(col)

In [ ]:

# --- helper: parse "Paper (Year)" out of your existing label strings ---
_paper_re = re.compile(r"\(([^()]+?\d{4})\)\s*$")  # grabs content in last "(...)" that ends with a year


def extract_paper_id(label: str) -> str:
    """
    Returns something stable like:
      'Kobulnicky & Kewley 2004'
      'Dopita et al. 2016'
    Falls back to full label if it can't parse.
    """
    m = _paper_re.search(label)
    return m.group(1).strip() if m else str(label).strip()


METALLICITY_COLUMN_INFO = {
    'Z_12logOH': ('KK04 iterative', 'R23/O32'),
    'Z_N2_Brazzini2024': ('Brazzini et al. 2024', 'N2'),
    'Z_O3N2_Brazzini2024': ('Brazzini et al. 2024', 'O3N2'),
    'Z_N2S2Halpha_Brazzini2024': ('Brazzini et al. 2024', 'N2S2Halpha'),
    'Z_R3_Brazzini2024': ('Brazzini et al. 2024', 'R3'),
    'Z_R23_Maiolino2008': ('Maiolino et al. 2008', 'R23'),
    'Z_N2_Maiolino2008': ('Maiolino et al. 2008', 'N2'),
    'Z_R23_Curti2017': ('Curti et al. 2017', 'R23'),
    'Z_R3_Curti2017': ('Curti et al. 2017', 'R3'),
    'Z_N2_Curti2017': ('Curti et al. 2017', 'N2'),
    'Z_O3N2_Curti2017': ('Curti et al. 2017', 'O3N2'),
    'Z_R_Pilyugin2016_highN2': ('Pilyugin et al. 2016 high N2', 'R'),
    'Z_R_Pilyugin2016_lowN2': ('Pilyugin et al. 2016 low N2', 'R'),
    'Z_S_Pilyugin2016_highN2': ('Pilyugin et al. 2016 high N2', 'S'),
    'Z_S_Pilyugin2016_lowN2': ('Pilyugin et al. 2016 low N2', 'S'),
    'Z_R23_KK2004': ('Kobulnicky & Kewley 2004', 'R23'),
    'Z_NII_KD2002': ('Kewley & Dopita 2002', 'NII'),
    'Z_D2016': ('Dopita et al. 2016', 'N2S2Halpha'),
    'Z_O3N2_M2013': ('Marino et al. 2013', 'O3N2'),
    'Z_N2_M2013': ('Marino et al. 2013', 'N2'),
    'Z_C2001': ('Charlot & Longhetti 2001', 'CL01'),
    'Z_N2_PP2004': ('Pettini \\& Pagel 2004', 'N2'),
    'Z_O3N2_PP2004': ('Pettini \\& Pagel 2004', 'O3N2'),
    'Z_N2_Brown2016': ('Brown et al. 2016', 'N2'),
    'Z_O3N2_Brown2016': ('Brown et al. 2016', 'O3N2'),
    'Z_N2O2_Brown2016': ('Brown et al. 2016', 'N2O2'),
    'Z_N2O2_KD2002': ('Kewley & Dopita 2002', 'N2O2'),
}


def _infer_metallicity_column_info(col):
    stem = col.removeprefix('Z_')
    parts = stem.split('_')
    if len(parts) == 1:
        return (parts[0], parts[0])
    indicator = parts[0]
    reference = ' '.join(parts[1:])
    return (reference, indicator)


def build_metallicity_calibrations(df, include_all=True):
    """Build the calibration registry for every saved metallicity column in the catalog."""
    nan_series = lambda: pd.Series(np.nan, index=df.index)
    cols = [
        col for col in df.columns
        if col.startswith('Z_') and not col.endswith('_e') and f'{col}_e' in df.columns
    ]
    if not include_all:
        cols = [col for col in METALLICITY_COLUMN_INFO if col in df.columns]
    ordered = [col for col in METALLICITY_COLUMN_INFO if col in cols]
    ordered += sorted(col for col in cols if col not in METALLICITY_COLUMN_INFO)
    calibrations = []
    for col in ordered:
        label, line_group = METALLICITY_COLUMN_INFO.get(col, _infer_metallicity_column_info(col))
        calibrations.append((df[col], df.get(f'{col}_e', nan_series()), label, line_group, col))
    return calibrations


# --- unified calibration registry ---
# Each entry: (data_array, error_array, label, line_group, column_name)
calibrations = build_metallicity_calibrations(cat)


In [ ]:
from itertools import cycle

def build_style_map(calibrations):
    paper_ids = []
    for item in calibrations:
        label = item[2]
        pid = extract_paper_id(label)
        if pid not in paper_ids:
            paper_ids.append(pid)
    print(len(paper_ids), "unique papers found.")
    cmap = plt.get_cmap('rainbow')
    colors = cmap(np.linspace(0, 1, 8))
    marker_cycle = cycle(['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'h', '8'])

    style_map = {}
    for i, pid in enumerate(paper_ids):
        style_map[pid] = {
            "color": colors[i % len(colors)],
            "marker": next(marker_cycle),
        }
    return style_map

style_map = build_style_map(calibrations)


In [ ]:
for col in cat.columns:
    print(col)

In [ ]:
# print(np.nanmean(cat['Z_O3N2_PP2004_e']))
# print(np.nanmean(cat['Z_O3N2_Brown2016_e']))
# print(np.nanmean(cat['Z_N2O2_Brown2016_e']))


In [ ]:
def fit_line_and_plot(
    x, y, label, *,
    yerr=None,
    do_plot=True,
    alpha=0.1,
    s=30,
    color=None,
    marker='o',
    shade_alpha=0.25,
    outline_lw=5,
    line_lw=3,
    bins=None,
    plot_bins=True,
    bin_min_count=1,
    bin_stat='median',
    bin_percentiles=(16, 84),
    bin_marker='h',
    bin_s=180,
    bin_edgecolor='k',
    bin_lw=1.5,
    capsize=4,
    capthick=1.5,
    fit_all_points=True,
    fit_binned=False,
    fit_range=None,
    fit_binned_range=None,
    fit_logy_points=False,
    fit_logy_bins=False,
    log_base=10,
    scatter_hollow=True,
    show_band=True,
    show_point_errors=False,
    legend=True,
    legend_loc='upper right',
    legend_frame=True,
):
    import numpy as np
    import matplotlib.pyplot as plt

    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if yerr is None:
        yerr = np.full_like(y, np.nan, dtype=float)
    else:
        yerr = np.asarray(yerr, float)

    def _apply_log(arr, log_base=10):
        if log_base == 10:
            return np.log10(arr)
        elif log_base in ['e', np.e]:
            return np.log(arr)
        raise ValueError("log_base must be 10 or 'e'")

    def _transform_yerr(yf, yerrf, fit_logy=False, log_base=10):
        if not fit_logy:
            return yerrf
        with np.errstate(divide='ignore', invalid='ignore'):
            if log_base == 10:
                return yerrf / (np.log(10.0) * yf)
            return yerrf / yf

    def _fit_linear(xf, yf, yerrf=None, fit_logy=False, log_base=10):
        xf = np.asarray(xf, float)
        yf = np.asarray(yf, float)
        if yerrf is None:
            yerrf = np.full_like(yf, np.nan, dtype=float)
        else:
            yerrf = np.asarray(yerrf, float)

        ok = np.isfinite(xf) & np.isfinite(yf)
        xf = xf[ok]
        yf = yf[ok]
        yerrf = yerrf[ok]

        if fit_logy:
            pos = yf > 0
            xf = xf[pos]
            yerrf = yerrf[pos]
            yf = yf[pos]
            if len(yf) < 2:
                return np.nan, np.nan, xf, yf, yerrf
            yerrf = _transform_yerr(yf, yerrf, fit_logy=True, log_base=log_base)
            yf = _apply_log(yf, log_base=log_base)

        if len(xf) < 2:
            return np.nan, np.nan, xf, yf, yerrf

        good_err = np.isfinite(yerrf) & (yerrf > 0)
        if good_err.sum() >= 2:
            m, b = np.polyfit(xf[good_err], yf[good_err], 1, w=1.0 / yerrf[good_err])
        else:
            m, b = np.polyfit(xf, yf, 1)
        return m, b, xf, yf, yerrf

    def _eval_fit(m, b, xgrid, fit_logy=False, log_base=10):
        if fit_logy:
            y_model = m * xgrid + b
            if log_base == 10:
                return 10**y_model
            elif log_base in ['e', np.e]:
                return np.exp(y_model)
        return m * xgrid + b

    def _fit_label(m, b, which='points', fit_logy=False, log_base=10, weighted=False):
        suffix = "weighted" if weighted else which
        if fit_logy:
            if log_base == 10:
                return fr'$\log(y) = {m:.2f}x {b:+.2f}$ ({suffix})'
            return fr'$\ln(y) = {m:.2f}x {b:+.2f}$ ({suffix})'
        return fr'$y = {m:.2f}x {b:+.2f}$ ({suffix})'

    msk = np.isfinite(x) & np.isfinite(y)
    if msk.sum() < 2:
        if do_plot:
            plt.scatter(x, y, alpha=alpha, s=s, color=color)
        return {
            'point_fit': (np.nan, np.nan),
            'binned_fit': (np.nan, np.nan),
            'bin_centers': None,
            'bin_values': None,
            'bin_yerr_low': None,
            'bin_yerr_high': None,
            'bin_counts': None,
        }

    xf = x[msk]
    yf = y[msk]
    yef = yerr[msk]

    bin_centers = None
    bin_values = None
    bin_yerr_low = None
    bin_yerr_high = None
    bin_counts = None
    binned_fit = (np.nan, np.nan)

    if bins is not None and plot_bins:
        bins = np.asarray(bins, float)
        bin_centers_ = 0.5 * (bins[:-1] + bins[1:])
        vals, lo, hi, cnt = [], [], [], []
        for i in range(len(bins) - 1):
            in_bin = (xf >= bins[i]) & (xf < bins[i + 1])
            nbin = np.sum(in_bin)
            cnt.append(nbin)
            if nbin >= bin_min_count:
                yb = yf[in_bin]
                yeb = yef[in_bin]
                if bin_stat == 'median':
                    y0 = np.median(yb)
                else:
                    raise ValueError("Currently only bin_stat='median' is supported.")
                p_lo, p_hi = np.percentile(yb, bin_percentiles)
                base_lo = y0 - p_lo
                base_hi = p_hi - y0
                finite_yeb = yeb[np.isfinite(yeb) & (yeb > 0)]
                med_err = np.median(finite_yeb) if finite_yeb.size else 0.0
                vals.append(y0)
                lo.append(np.sqrt(base_lo**2 + med_err**2))
                hi.append(np.sqrt(base_hi**2 + med_err**2))
            else:
                vals.append(np.nan)
                lo.append(np.nan)
                hi.append(np.nan)
        bin_centers = np.asarray(bin_centers_)
        bin_values = np.asarray(vals)
        bin_yerr_low = np.asarray(lo)
        bin_yerr_high = np.asarray(hi)
        bin_counts = np.asarray(cnt)

    if do_plot and show_point_errors:
        valid_err = np.isfinite(yef) & (yef > 0)
        if np.any(valid_err):
            plt.errorbar(xf[valid_err], yf[valid_err], yerr=yef[valid_err], fmt='none', ecolor=color, alpha=max(alpha, 0.15), elinewidth=0.8, capsize=0, zorder=0)

    if do_plot:
        scatter_kwargs = dict(alpha=alpha, s=s, marker=marker, zorder=1)
        if scatter_hollow:
            scatter_kwargs.update(facecolors='none', edgecolors=color, linewidths=1.0)
        else:
            scatter_kwargs.update(c=color, edgecolors=color, linewidths=0.1)
        plt.scatter(xf, yf, **scatter_kwargs)

    point_fit = (np.nan, np.nan)
    if fit_all_points:
        xfit_data = xf.copy()
        yfit_data = yf.copy()
        yerr_data = yef.copy()
        if fit_range is not None:
            xmin, xmax = fit_range
            keep = (xfit_data >= xmin) & (xfit_data <= xmax)
            xfit_data = xfit_data[keep]
            yfit_data = yfit_data[keep]
            yerr_data = yerr_data[keep]
        m, b, xfit_use, yfit_use, yerr_use = _fit_linear(xfit_data, yfit_data, yerr_data, fit_logy=fit_logy_points, log_base=log_base)
        point_fit = (m, b)
        if do_plot and np.isfinite(m) and np.isfinite(b):
            xline = np.linspace(np.nanmin(xfit_data if len(xfit_data) else xf), np.nanmax(xfit_data if len(xfit_data) else xf), 300)
            yline = _eval_fit(m, b, xline, fit_logy=fit_logy_points, log_base=log_base)
            if show_band and (not fit_logy_points) and len(xfit_use) > 2:
                n = len(xfit_use)
                xbar = np.mean(xfit_use)
                yfit_pts = m * xfit_use + b
                s_err = np.sqrt(np.sum((yfit_use - yfit_pts) ** 2) / (n - 2))
                Sxx = np.sum((xfit_use - xbar) ** 2)
                if Sxx > 0:
                    y_err = s_err * np.sqrt(1 / n + (xline - xbar) ** 2 / Sxx)
                    plt.fill_between(xline, yline - y_err, yline + y_err, color=color, alpha=shade_alpha, zorder=2)
            plt.plot(xline, yline, color='black', linewidth=outline_lw, linestyle='-', zorder=3)
            weighted = np.isfinite(yerr_use).sum() >= 2
            plt.plot(xline, yline, color=color, linewidth=line_lw, linestyle='-', label=_fit_label(m, b, which='points', fit_logy=fit_logy_points, log_base=log_base, weighted=weighted), zorder=4)

    if do_plot and (bin_centers is not None):
        valid_bins = np.isfinite(bin_centers) & np.isfinite(bin_values) & np.isfinite(bin_yerr_low) & np.isfinite(bin_yerr_high)
        if np.any(valid_bins):
            plt.errorbar(bin_centers[valid_bins], bin_values[valid_bins], yerr=[bin_yerr_low[valid_bins], bin_yerr_high[valid_bins]], fmt='none', ecolor='k', elinewidth=1.5, capsize=capsize, capthick=capthick, zorder=5)
            plt.scatter(bin_centers[valid_bins], bin_values[valid_bins], s=bin_s, marker=bin_marker, facecolors=color, edgecolors=bin_edgecolor, linewidths=bin_lw, zorder=6, label=f'{label} bins')

    if fit_binned and (bin_centers is not None) and (bin_values is not None):
        valid_bins = np.isfinite(bin_centers) & np.isfinite(bin_values)
        xb = bin_centers[valid_bins]
        yb = bin_values[valid_bins]
        yeb = 0.5 * (bin_yerr_low[valid_bins] + bin_yerr_high[valid_bins]) if bin_yerr_low is not None else np.full_like(yb, np.nan)
        if fit_binned_range is not None:
            xmin, xmax = fit_binned_range
            keep = (xb >= xmin) & (xb <= xmax)
            xb = xb[keep]
            yb = yb[keep]
            yeb = yeb[keep]
        mb, bb, xb_use, yb_use, yeb_use = _fit_linear(xb, yb, yeb, fit_logy=fit_logy_bins, log_base=log_base)
        binned_fit = (mb, bb)
        if do_plot and np.isfinite(mb) and np.isfinite(bb):
            xline_b = np.linspace(np.nanmin(xb if len(xb) else bin_centers[valid_bins]), np.nanmax(xb if len(xb) else bin_centers[valid_bins]), 300)
            yline_b = _eval_fit(mb, bb, xline_b, fit_logy=fit_logy_bins, log_base=log_base)
            weighted = np.isfinite(yeb_use).sum() >= 2
            plt.plot(xline_b, yline_b, color='k', linestyle='--', linewidth=2.5, label=_fit_label(mb, bb, which='bins', fit_logy=fit_logy_bins, log_base=log_base, weighted=weighted), zorder=7)

    if do_plot and legend:
        leg = plt.legend(loc=legend_loc, frameon=legend_frame, framealpha=1.0, edgecolor='k')
        if leg is not None:
            leg.get_frame().set_linewidth(1.5)

    return {
        'point_fit': point_fit,
        'binned_fit': binned_fit,
        'bin_centers': bin_centers,
        'bin_values': bin_values,
        'bin_yerr_low': bin_yerr_low,
        'bin_yerr_high': bin_yerr_high,
        'bin_counts': bin_counts,
    }


In [ ]:
# Literature M33 gradients overplotted on metallicity calibration panels.
literature_gradients = [
    dict(label='Rosolowsky+2008', m=-0.027, b=8.36, ls='-'),
    dict(label='Magrini+2010', m=-0.044, b=8.447, ls='--'),
    dict(label='Rogers+2022', m=-0.037, b=8.59, ls='-.'),
]

PAPER_METALLICITY_SELECTION = [
    'Z_O3N2_M2013',
    'Z_N2S2Halpha_Brazzini2024',
    'Z_N2_M2013',
    'Z_R3_Brazzini2024',
    'Z_N2O2_Brown2016',
]


def _calibration_col(item):
    return item[4] if len(item) > 4 else None


def _select_calibrations(calibrations, selected_columns=None):
    if selected_columns is None:
        return list(calibrations)
    selected = []
    wanted = list(selected_columns)
    for col in wanted:
        match = [item for item in calibrations if _calibration_col(item) == col]
        if match:
            selected.append(match[0])
        else:
            print(f'Warning: requested metallicity calibration column not found: {col}')
    return selected


def _metallicity_fit_with_covariance(x, y, yerr=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if yerr is None:
        yerr = np.full_like(y, np.nan, dtype=float)
    else:
        yerr = np.asarray(yerr, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]
    yerr = yerr[ok]
    if len(x) < 2:
        return np.nan, np.nan, np.nan, np.nan

    good_err = np.isfinite(yerr) & (yerr > 0)
    x_fit = x[good_err] if good_err.sum() >= 2 else x
    y_fit = y[good_err] if good_err.sum() >= 2 else y
    weights = 1.0 / yerr[good_err] if good_err.sum() >= 2 else None
    if len(x_fit) < 2:
        return np.nan, np.nan, np.nan, np.nan

    m, b = np.polyfit(x_fit, y_fit, 1, w=weights)
    m_err = np.nan
    b_err = np.nan
    if len(x_fit) > 2:
        try:
            _, cov = np.polyfit(x_fit, y_fit, 1, w=weights, cov=True)
            m_err = float(np.sqrt(cov[0, 0]))
            b_err = float(np.sqrt(cov[1, 1]))
        except Exception:
            reg = linregress(x_fit, y_fit)
            m_err = reg.stderr
            b_err = reg.intercept_stderr
    return float(m), float(b), float(m_err), float(b_err)


def _metallicity_binned_stats(x, y, yerr, bins, min_count=5):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    yerr = np.asarray(yerr, dtype=float)
    centers = 0.5 * (bins[:-1] + bins[1:])
    med = np.full(len(centers), np.nan)
    lo = np.full(len(centers), np.nan)
    hi = np.full(len(centers), np.nan)
    counts = np.zeros(len(centers), dtype=int)

    for i in range(len(centers)):
        in_bin = np.isfinite(x) & np.isfinite(y) & (x >= bins[i]) & (x < bins[i + 1])
        counts[i] = int(in_bin.sum())
        if counts[i] < min_count:
            continue
        yb = y[in_bin]
        yeb = yerr[in_bin]
        y0 = np.nanmedian(yb)
        p16 = np.nanpercentile(yb, 16)
        p84 = np.nanpercentile(yb, 84)
        finite_err = yeb[np.isfinite(yeb) & (yeb > 0)]
        med_err = np.nanmedian(finite_err) if finite_err.size else 0.0
        med[i] = y0
        lo[i] = np.sqrt((y0 - p16) ** 2 + med_err ** 2)
        hi[i] = np.sqrt((p84 - y0) ** 2 + med_err ** 2)
    return centers, med, lo, hi, counts


def _average_metallicity_error(yerr):
    yerr = np.asarray(yerr, dtype=float)
    good = yerr[np.isfinite(yerr) & (yerr > 0)]
    return float(np.nanmedian(good)) if len(good) else np.nan


def _calibration_display_name(label, line_group):
    clean = _clean_calibration_label(label) if '_clean_calibration_label' in globals() else str(label).replace('\\&', '&')
    return f'{line_group}: {clean}'


def _metallicity_color_map(calibrations):
    cmap = plt.get_cmap('rainbow')
    # Use only the blue-purple portion of rainbow, kept separate from the green-red property palette.
    colors = cmap(np.linspace(0.02, 0.30, max(len(calibrations), 2)))
    return {_calibration_col(item) or f'{item[3]}_{item[2]}': colors[i] for i, item in enumerate(calibrations)}


def plot_metallicity_calibration_multipanel(
    calibrations,
    style_map,
    R_gal,
    save_subdir='metallicity_calibrations',
    filename='M33_metallicity_calibrations_multipanel.png',
    fit_range=(0, 8),
    bins=None,
    ylim=(8, 9),
    selected_columns=None,
    show_histograms=True,
    show=True,
):
    """Plot one wide stacked metallicity-gradient panel per calibration with side histograms."""
    mpl.rcParams['text.usetex'] = False
    if bins is None:
        bins = np.arange(fit_range[0], fit_range[1] + 0.5, 0.5)

    selected_calibrations = _select_calibrations(calibrations, selected_columns)
    if not selected_calibrations:
        raise RuntimeError('No metallicity calibrations were selected for plotting.')

    x = np.asarray(R_gal, dtype=float)
    xgrid = np.linspace(fit_range[0], fit_range[1], 300)
    available = []
    for item in selected_calibrations:
        arr, err, label, grp = item[:4]
        col = _calibration_col(item)
        y = np.asarray(arr, dtype=float)
        yerr = np.asarray(err, dtype=float) if err is not None else np.full_like(y, np.nan, dtype=float)
        mask = np.isfinite(x) & np.isfinite(y) & (x >= fit_range[0]) & (x <= fit_range[1])
        if mask.sum() < 2:
            continue
        available.append((y, yerr, label, grp, col, mask))

    if not available:
        raise RuntimeError('No metallicity calibrations have enough finite data to plot.')

    if selected_columns is None:
        group_order = ['R23/O32', 'R23', 'R', 'S', 'O3N2', 'N2S2Halpha', 'N2', 'R3', 'N2O2', 'NII', 'CL01']
        group_rank = {grp: i for i, grp in enumerate(group_order)}
        available.sort(key=lambda item: (group_rank.get(item[3], 99), item[2]))

    color_by_col = _metallicity_color_map([(None, None, label, grp, col) for _, _, label, grp, col, _ in available])

    fig_height = max(2.45 * len(available), 8.5)
    fig = plt.figure(figsize=(13.8, fig_height))
    gs = fig.add_gridspec(
        nrows=len(available), ncols=2,
        width_ratios=[5.6, 1.08],
        hspace=0.06, wspace=0.025,
    )
    main_axes = []
    hist_axes = []

    for i, (y, yerr, label, grp, col, mask) in enumerate(available):
        ax = fig.add_subplot(gs[i, 0], sharex=main_axes[0] if main_axes else None, sharey=main_axes[0] if main_axes else None)
        hax = fig.add_subplot(gs[i, 1], sharey=ax)
        main_axes.append(ax)
        hist_axes.append(hax)

        color = color_by_col[col]
        pid = extract_paper_id(label)
        marker = style_map.get(pid, {}).get('marker', 'o')

        ax.scatter(
            x[mask], y[mask],
            s=11, alpha=0.14, facecolors='none', edgecolors=color,
            linewidths=0.5, marker=marker, zorder=1,
        )

        centers, med, lo, hi, counts = _metallicity_binned_stats(x[mask], y[mask], yerr[mask], bins, min_count=5)
        valid = np.isfinite(med)
        ax.errorbar(
            centers[valid], med[valid],
            yerr=[lo[valid], hi[valid]],
            fmt='none', ecolor='k', elinewidth=1.0, capsize=2.5, capthick=1.0,
            zorder=3,
        )
        ax.scatter(
            centers[valid], med[valid],
            s=44, marker='D', facecolors=color, edgecolors='k', linewidths=0.75,
            zorder=4,
        )

        slope, intercept, slope_err, _ = _metallicity_fit_with_covariance(x[mask], y[mask], yerr[mask])
        if np.isfinite(slope) and np.isfinite(intercept):
            ax.plot(xgrid, slope * xgrid + intercept, color=color, lw=2.4, zorder=5)

        for lit in literature_gradients:
            ax.plot(
                xgrid, lit['b'] + lit['m'] * xgrid,
                color='k', linestyle=lit['ls'], linewidth=1.05, alpha=0.75,
                zorder=2,
            )

        slope_text = f'slope = {slope:.3f}'
        if np.isfinite(slope_err):
            slope_text += f' $\\pm$ {slope_err:.3f}'
        slope_text += r' dex kpc$^{-1}$'
        ax.text(
            0.02, 0.91,
            f'{_calibration_display_name(label, grp)}\n{slope_text}',
            transform=ax.transAxes,
            ha='left', va='top', fontsize=12.5,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.72),
            zorder=8,
        )

        avg_err = _average_metallicity_error(yerr[mask])
        if np.isfinite(avg_err):
            xerr_pt = fit_range[1] - 0.45
            yerr_pt = ylim[1] - 0.16
            ax.errorbar(
                [xerr_pt], [yerr_pt], yerr=[[avg_err], [avg_err]],
                fmt='o', ms=4.5, color=color, ecolor='k', elinewidth=1.0,
                capsize=2.5, zorder=8,
            )
            ax.text(
                0.965, 0.84, r'$\tilde{\sigma}_Z$=' + f'{avg_err:.2g}',
                transform=ax.transAxes, ha='right', va='center', fontsize=10.5,
                bbox=dict(boxstyle='round,pad=0.16', facecolor='white', edgecolor='none', alpha=0.65),
                zorder=8,
            )

        data = y[mask]
        data = data[np.isfinite(data)]
        if show_histograms and len(data):
            hax.hist(data, bins=np.linspace(ylim[0], ylim[1], 34), orientation='horizontal', color=color, alpha=0.24, edgecolor='k', linewidth=0.45)
            median = np.nanmedian(data)
            p16 = np.nanpercentile(data, 16)
            p84 = np.nanpercentile(data, 84)
            handles = [
                hax.axhline(median, color=color, linestyle='-', lw=1.5, label=f'Med. {median:.2f}'),
                hax.axhline(p16, color=color, linestyle='--', lw=1.1, label=f'16th {p16:.2f}'),
                hax.axhline(p84, color=color, linestyle='--', lw=1.1, label=f'84th {p84:.2f}'),
            ]
            hax.legend(handles=handles, loc='upper right', frameon=False, fontsize=7.2, handlelength=1.25, borderpad=0.15, labelspacing=0.12)
        hax.set_xlabel('$N$', fontsize=9.5)
        hax.tick_params(axis='x', labelsize=8.5)
        hax.tick_params(axis='y', labelleft=False, labelright=False)
        hax.set_ylim(*ylim)
        hax.minorticks_on()

        ax.set_xlim(*fit_range)
        ax.set_ylim(*ylim)
        ax.minorticks_on()
        ax.tick_params(axis='both', labelsize=10.5)
        ax.set_ylabel('12 + log(O/H)', fontsize=12)
        if i < len(available) - 1:
            ax.tick_params(labelbottom=False)

    main_axes[-1].set_xlabel('Galactocentric Radius (kpc)', fontsize=13)

    calibration_handles = [
        mpl.lines.Line2D([], [], color=color_by_col[col], marker='D', linestyle='none', markersize=7, label=_calibration_display_name(label, grp))
        for _, _, label, grp, col, _ in available
    ]
    literature_handles = [
        mpl.lines.Line2D([], [], color='k', linestyle=lit['ls'], linewidth=1.4, label=lit['label'])
        for lit in literature_gradients
    ]
    max_handles = 12 if len(calibration_handles) > 8 else len(calibration_handles)
    legend_handles = calibration_handles[:max_handles] + literature_handles
    if len(calibration_handles) > max_handles:
        legend_handles.insert(max_handles, mpl.lines.Line2D([], [], linestyle='none', label=f'+ {len(calibration_handles) - max_handles} more calibrations'))
    fig.legend(
        handles=legend_handles,
        loc='upper center', bbox_to_anchor=(0.5, 0.996),
        ncol=4,
        frameon=False, fontsize=11.5,
    )
    fig.subplots_adjust(top=0.965 if len(available) > 8 else 0.93)

    outpath = paper_plot_path(filename, subdir=save_subdir)
    fig.savefig(outpath, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return outpath, main_axes, hist_axes


# Backwards-compatible wrapper name; now writes one stacked multi-panel figure per catalog subset.
def plot_calibration_panel(line_group, calibrations, style_map, R_gal, save_subdir='metallicity_calibrations', filename_prefix='', show=True):
    return plot_metallicity_calibration_multipanel(
        calibrations,
        style_map,
        R_gal,
        save_subdir=save_subdir,
        filename=f'{filename_prefix}M33_metallicity_calibrations_multipanel.png',
        show=show,
    )


def plot_metallicity_vs_ionization_pressure_grid(
    df,
    calibrations,
    style_map,
    selected_columns=None,
    save_subdir='metallicity_calibrations',
    filename='M33_metallicity_vs_logU_pressure_grid.png',
    show=True,
):
    """Plot metallicity calibrations versus logU and log(P/k) in a two-column grid."""
    mpl.rcParams['text.usetex'] = False
    selected_calibrations = _select_calibrations(calibrations, selected_columns)
    selected_calibrations = [item for item in selected_calibrations if _calibration_col(item) in df.columns]
    if not selected_calibrations:
        raise RuntimeError('No metallicity calibrations were selected for the logU/pressure grid.')

    # Match the purple-blue colour assignment used by the metallicity radial-gradient panels.
    color_by_col = _metallicity_color_map(selected_calibrations)

    x_specs = [
        ('logU_KK04', r'$\log U$'),
        ('log_P_thermal_SII_over_k', r'$\log_{10}(P/k\,[{\rm K\,cm}^{-3}])$'),
    ]
    nrows = len(selected_calibrations)
    fig, axes = plt.subplots(nrows, 2, figsize=(13.5, max(2.2 * nrows, 5.5)), sharex='col', sharey=True)
    axes = np.atleast_2d(axes)

    for row, item in enumerate(selected_calibrations):
        y_arr, y_err, label, grp = item[:4]
        col = _calibration_col(item)
        display = _calibration_display_name(label, grp)
        color = color_by_col.get(col, 'C0')
        y = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
        for j, (xcol, xlabel) in enumerate(x_specs):
            ax = axes[row, j]
            if xcol not in df.columns:
                ax.text(0.5, 0.5, f'Missing {xcol}', transform=ax.transAxes, ha='center', va='center')
                ax.set_axis_off()
                continue
            x = pd.to_numeric(df[xcol], errors='coerce').to_numpy(dtype=float)
            mask = np.isfinite(x) & np.isfinite(y) & (y >= 8.0) & (y <= 9.0)
            ax.scatter(x[mask], y[mask], s=9, alpha=0.18, color=color, edgecolors='none')
            if mask.sum() >= 3:
                fit = linregress(x[mask], y[mask])
                xgrid = np.linspace(np.nanmin(x[mask]), np.nanmax(x[mask]), 200)
                ax.plot(xgrid, fit.slope * xgrid + fit.intercept, color='k', lw=2.0,
                        label=rf'slope={fit.slope:.3f}$\pm${fit.stderr:.3f}')
                ax.legend(frameon=False, fontsize=10, loc='upper right')
            ax.set_ylim(8, 9)
            ax.minorticks_on()
            if row == nrows - 1:
                ax.set_xlabel(xlabel)
            if j == 0:
                ax.set_ylabel(r'$12+\log({\rm O/H})$')
                ax.text(0.02, 0.93, display, transform=ax.transAxes, ha='left', va='top', fontsize=11,
                        bbox=dict(facecolor='white', alpha=0.70, edgecolor='none', pad=2))
    fig.tight_layout()
    outpath = paper_plot_path(filename, subdir=save_subdir)
    fig.savefig(outpath, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return outpath, axes


In [ ]:
fig_m33_ratio_mosaics, axes_m33_ratio_mosaics = plot_region_panel_m33_mosaics(
    m33_ratio_mosaic_panel_specs,
    field_order=mosaic_field_order,
    maps_dir='../M33-Maps-Calibrated',
    field_valid_bounds=(50, 2000, 50, 2000),
    display_stride=4,
    ncols=2,
    figsize=(14, 18),
    axis_label_fontsize=18,
    tick_label_fontsize=13,
    colorbar_fontsize=15,
    colorbar_tick_labelsize=12,
    wspace=0.08,
    hspace=0.08,
    output_path=str(paper_plot_path('M33_full_mosaic_ratio_panels_dered.png', subdir='region_panels')),
    dpi=300,
)

## Cells From 8b Not Used By Paper Notebook Or BPT Comparison

In [ ]:
# save the catalog to a latex table

In [ ]:
# Initialize values that will be extended with fit results later in the notebook.
values, formats = build_catalog_number_values(cat, snr_cut=3)
add_duplicate_region_values(values, cat_before_primary_filter)
if {'has_snr_in_boundary', 'has_wr_in_boundary'}.issubset(cat.columns):
    snr_hosts = cat['has_snr_in_boundary'].fillna(False).astype(bool)
    wr_hosts = cat['has_wr_in_boundary'].fillna(False).astype(bool)
    values['nRegionsContainingSNR'] = int(snr_hosts.sum())
    values['nRegionsContainingWR'] = int(wr_hosts.sum())
    values['nRegionsContainingSNRAndWR'] = int((snr_hosts & wr_hosts).sum())
    values['nRegionsContainingSNROrWR'] = int((snr_hosts | wr_hosts).sum())
if 'n_pn_in_boundary' in cat.columns:
    values['nCrossMatchedPN'] = int(pd.to_numeric(cat['n_pn_in_boundary'], errors='coerce').fillna(0).sum())
radial_gradient_fit_results = {}

print("\nCatalog and peak-removal commands initialized:")
for k in values:
    print(f"\\{k} = {format_value(values[k], formats.get(k))}")


In [ ]:
# Optional quick-look velocity-dispersion histogram.
if 'sigma_mean_kms' in cat.columns:
    sigma_values = pd.to_numeric(cat['sigma_mean_kms'], errors='coerce').to_numpy(dtype=float)
    sigma_values = sigma_values[np.isfinite(sigma_values)]
    plt.hist(sigma_values, bins=100, density=True, color='lightgray', edgecolor='black')
    plt.xlabel(r'$\sigma$ (km s$^{-1}$)')
    plt.ylabel('Density')
    plt.show()
else:
    print('sigma_mean_kms is not available yet. Run the final notebook 7 kinematic update cell first.')



In [ ]:
cat.columns

In [ ]:
for col in cat.columns:
    print(col)

In [ ]:
# number of selected star-forming regions
sf_count = int(selected_star_forming_mask(cat).sum())
print(f"Star-forming definition: {star_forming_definition_label()}")
print(f'Number of selected star-forming regions: {sf_count} out of {len(cat)}')

In [ ]:
np.nanmin(cat['SNR_Halpha_sum'])

#how many regions have SNR_Halpha_sum <= 3.0?
low_snr_count = np.sum(cat['SNR_Halpha_sum'] <= 3.0)
print(f"Number of regions with SNR_Halpha_sum <= 3.0: {low_snr_count}")

In [ ]:
# min(cat['area_px_after_carve'])

#how many regions have area_px_after_carve <= 20 and low Halpha flux?
small_region_count = np.sum((cat['area_px_after_carve'] <= 20))
print(f"Number of regions with area_px_after_carve <= 20 and F_Halpha <= 10.0: {small_region_count}")
#which field are these regions in?
small_regions = cat[(cat['area_px_after_carve'] <= 20)]
print(small_regions[['field', 'area_px_after_carve', 'F_Halpha_sum_dered']])

In [ ]:
plt.hist(cat["F_Halpha_sum_dered"], bins=50, range=(1e-18, 1e-12), log=True)
# plt.xscale('log')
plt.show()

In [ ]:
# plot radius vs luminosity
#don't include regions with tiny radius
mask = (cat['radius_p16_pc'] > 1) & (cat['radius_p50_pc'] > 1) & (cat['radius_p84_pc'] > 1)

plt.figure(figsize=(10, 6))
# plt.scatter(cat['radius_p84_pc'][mask], cat['L_Ha_sum_dered'][mask], s=20, alpha=0.7, edgecolor='k', color = 'purple', label = r'84\% radius')

selected_sf_radius_mask = selected_star_forming_mask(cat)
plt.scatter(cat['radius_p50_pc'][mask & selected_sf_radius_mask], 
            cat['L_Ha_sum_dered'][mask & selected_sf_radius_mask], 
            s=20, alpha=1, edgecolor='k', color = 'skyblue', marker='o', label='Selected star-forming')

plt.scatter(cat['radius_p50_pc'][mask & (cat['BPT_class_sum_dered'] == 'Composite')], 
            cat['L_Ha_sum_dered'][mask & (cat['BPT_class_sum_dered'] == 'Composite')], 
            s=40, alpha=1, edgecolor='k', color = 'green', marker='^', label='Composite')

plt.scatter(cat['radius_p50_pc'][mask & (cat['BPT_class_sum_dered'] == 'AGN/Shock')], 
            cat['L_Ha_sum_dered'][mask & (cat['BPT_class_sum_dered'] == 'AGN/Shock')], 
            s=40, alpha=1, edgecolor='k', color = 'red', marker='s', label='AGN/Shock')

# plt.scatter(cat['radius_p16_pc'][mask], cat['L_Ha_sum_dered'][mask], s=20, alpha=0.7, edgecolor='k', color = 'hotpink', label = r'16\% radius')
plt.yscale('log')
plt.xlabel('Region Size (pc)')
plt.ylabel(r'H$\alpha$ Luminosity (erg/s)')
plt.minorticks_on()
plt.legend()
plt.savefig(paper_plot_path('M33_HII_luminosity_vs_radius_BPT.png', subdir='scaling_relations'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Shared color dictionaries are initialized near the top of the notebook so early plots can use them.
# Recompute here as a harmless refresh before the radial-gradient section.
property_cmap = plt.get_cmap('rainbow')
property_color_positions = {
    'sum_A_V': 0.38,
    'L_Ha_sum_dered': 0.52,
    'ne_SII_cm3': 0.66,
    'logU_KK04': 0.76,
    'log_P_thermal_SII_over_k': 0.86,
    'radius_areaeq_pc': 0.94,
}
color_dic = {key: property_cmap(pos) for key, pos in property_color_positions.items()}
lf_comparison_colors = {
    'pre_dig': property_cmap(0.48),
    'dig_corrected': color_dic['L_Ha_sum_dered'],
}


In [ ]:
for col in cat.columns:
    print(col)

In [ ]:
#number of regions with Ha luminosity above 1e38.6 erg/s
n_high_luminosity = np.sum(cat['L_Ha_sum_dered'] > 10**38.6)
print(f'Number of regions with L_Ha_sum_dered > 1e38.6 erg/s: {n_high_luminosity}')


In [ ]:
for col in cat.columns:
    print(col)

In [ ]:
# print(np.nanmean(cat['Z_O3N2_PP2004_e']))
# print(np.nanmean(cat['Z_O3N2_Brown2016_e']))
# print(np.nanmean(cat['Z_N2O2_Brown2016_e']))


In [ ]:
# metallicity_all_multipanel_path, metallicity_all_multipanel_axes, metallicity_all_multipanel_hist_axes, metallicity_all_multipanel_fit_results = plot_metallicity_calibration_multipanel(
#     calibrations,
#     style_map,
#     cat['R_gal_kpc'],
#     save_subdir='metallicity_calibrations',
#     filename='M33_metallicity_calibrations_all_indicators_multipanel.png',
# )
# print(f'Wrote all-indicator metallicity gradient multipanel: {metallicity_all_multipanel_path}')
# save_for_paper(
#     metallicity_all_multipanel_path,
#     'M33_metallicity_calibrations_all_indicators_multipanel.png',
# )

metallicity_paper_multipanel_path, metallicity_paper_multipanel_axes, metallicity_paper_multipanel_hist_axes, metallicity_paper_multipanel_fit_results = plot_metallicity_calibration_multipanel(
    calibrations,
    style_map,
    cat['R_gal_kpc'],
    save_subdir='metallicity_calibrations',
    filename='M33_metallicity_calibrations_paper_multipanel.png',
    selected_columns=PAPER_METALLICITY_SELECTION,
)
print(f'Wrote paper metallicity gradient multipanel: {metallicity_paper_multipanel_path}')

metallicity_multipanel_alias_path = paper_plot_path(
    'M33_metallicity_calibrations_multipanel.png',
    subdir='metallicity_calibrations',
)
shutil.copyfile(metallicity_paper_multipanel_path, metallicity_multipanel_alias_path)
print(f'Updated compatibility metallicity multipanel alias: {metallicity_multipanel_alias_path}')
print('Note: legacy per-indicator metallicity PNGs in this folder are not regenerated by the new multipanel code.')


metallicity_logu_pressure_grid_path, _, metallicity_logu_pressure_fit_results = plot_metallicity_vs_ionization_pressure_grid(
    cat,
    calibrations,
    style_map,
    selected_columns=PAPER_METALLICITY_SELECTION,
    save_subdir='metallicity_calibrations',
    filename='M33_metallicity_vs_logU_pressure_grid.png',
)
print(f'Wrote metallicity versus logU/pressure grid: {metallicity_logu_pressure_grid_path}')


In [ ]:
# plot DIG level vs Av
sel = (cat['sum_A_V'] < 3) & (cat['sum_A_V'] > 0) & np.isfinite(cat['sum_A_V']) & np.isfinite(cat['bg_local'])
# plt.scatter(cat['sum_A_V'][sel], cat['bg_local'][sel], alpha=0.2, s=10, color='C1')
# plt.yscale('log')

plot_radial_gradient(
    cat=cat,
    xcol='sum_A_V',
    ycol='bg_local',
    mask=sel,
    bins=np.arange(0, 3, 0.1),
    color='C1',
    xlabel=r'$A_V$ (mag)',
    ylabel='Local background (DIG proxy)',
    yscale='log',
    alpha=0.2,
    savepath=str(paper_plot_path('M33_HII_bg_local_vs_Av.png', subdir='dig_diagnostics'))
)


In [ ]:
# Multi-panel cutout plot for one example region with configurable line-ratio / line-map panels.

INTRINSIC_HA_HB_MAP = 2.86
R_V_MAP = 3.1


def ccm89_k_lambda_map(wave_angstrom, R_V=3.1):
    wave_micron = wave_angstrom * 1e-4
    x = 1.0 / wave_micron
    y = x - 1.82
    a = (
        1
        + 0.17699 * y
        - 0.50447 * y**2
        - 0.02427 * y**3
        + 0.72085 * y**4
        + 0.01979 * y**5
        - 0.77530 * y**6
        + 0.32999 * y**7
    )
    b = (
        1.41338 * y
        + 2.28305 * y**2
        + 1.07233 * y**3
        - 5.38434 * y**4
        - 0.62251 * y**5
        + 5.30260 * y**6
        - 2.09002 * y**7
    )
    return a * R_V + b


LINE_WAVELENGTHS_REGION = {
    'Hbflux': 4861.0,
    'OIII4959flux': 4959.0,
    'OIII5007flux': 5007.0,
    'Haflux': 6563.0,
    'NII6548flux': 6548.0,
    'NII6584flux': 6584.0,
    'NII6583flux': 6584.0,
    'SII6716flux': 6716.0,
    'SII6731flux': 6731.0,
    'OII3727flux': 3727.0,
}


def _normalize_region_identifier(field, region_id):
    field = str(field)
    if isinstance(region_id, str):
        rid = region_id.strip()
        if '_' in rid:
            region_label = int(rid.split('_')[-1])
            return rid, region_label
        region_label = int(float(rid))
        return f"{field}_{region_label:04d}", region_label
    region_label = int(region_id)
    return f"{field}_{region_label:04d}", region_label



def _sum_maps_from_spec(field, line_names, maps_dir, scale=1e-17, cache=None):
    if isinstance(line_names, (str, bytes)):
        line_names = [line_names]
    else:
        line_names = list(line_names)

    arrays = []
    for line_name in line_names:
        key = (field, line_name, scale)
        if cache is not None and key in cache:
            data = cache[key]
        else:
            map_path = Path(maps_dir) / f"M33-{field}" / f"M33{field}-{line_name}.fits"
            if not map_path.exists():
                raise FileNotFoundError(f"Missing map for {field} {line_name}: {map_path}")
            data = fits.getdata(map_path).astype(float)
            data = np.where(np.isfinite(data), data / scale, np.nan)
            if cache is not None:
                cache[key] = data
        arrays.append(data)

    return np.nansum(np.stack(arrays, axis=0), axis=0)



def _compute_ebv_map(field, maps_dir, cache=None, intrinsic_ha_hb=INTRINSIC_HA_HB_MAP, R_V=R_V_MAP):
    cache_key = ('EBV', field, intrinsic_ha_hb, R_V)
    if cache is not None and cache_key in cache:
        return cache[cache_key]

    ha = _sum_maps_from_spec(field, 'Haflux', maps_dir=maps_dir, cache=cache)
    hb = _sum_maps_from_spec(field, 'Hbflux', maps_dir=maps_dir, cache=cache)

    k_ha = ccm89_k_lambda_map(6563.0, R_V=R_V)
    k_hb = ccm89_k_lambda_map(4861.0, R_V=R_V)
    delta_k = k_hb - k_ha

    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = ha / hb
        ebv = (2.5 / delta_k) * np.log10(ratio / intrinsic_ha_hb)

    good = np.isfinite(ha) & np.isfinite(hb) & (ha > 0) & (hb > 0)
    ebv = np.where(good, ebv, np.nan)
    ebv = np.where(np.isfinite(ebv), np.clip(ebv, 0, None), np.nan)

    if cache is not None:
        cache[cache_key] = ebv
    return ebv



def _deredden_single_line_map(field, line_name, maps_dir, cache=None, scale=1e-17, intrinsic_ha_hb=INTRINSIC_HA_HB_MAP, R_V=R_V_MAP):
    wave = LINE_WAVELENGTHS_REGION.get(line_name)
    if wave is None:
        raise KeyError(f"No wavelength mapping configured for line {line_name}")

    flux_map = _sum_maps_from_spec(field, line_name, maps_dir=maps_dir, scale=scale, cache=cache)
    ebv_map = _compute_ebv_map(field, maps_dir=maps_dir, cache=cache, intrinsic_ha_hb=intrinsic_ha_hb, R_V=R_V)
    k_line = ccm89_k_lambda_map(wave, R_V=R_V)
    correction = 10.0 ** (0.4 * ebv_map * k_line)

    with np.errstate(invalid='ignore'):
        flux_map_dered = flux_map * correction
    flux_map_dered = np.where(np.isfinite(flux_map), flux_map_dered, np.nan)
    return flux_map_dered



def _sum_dereddened_maps(field, line_names, maps_dir, cache=None, scale=1e-17, intrinsic_ha_hb=INTRINSIC_HA_HB_MAP, R_V=R_V_MAP):
    if isinstance(line_names, (str, bytes)):
        line_names = [line_names]
    else:
        line_names = list(line_names)

    arrays = [
        _deredden_single_line_map(
            field,
            line_name,
            maps_dir=maps_dir,
            cache=cache,
            scale=scale,
            intrinsic_ha_hb=intrinsic_ha_hb,
            R_V=R_V,
        )
        for line_name in line_names
    ]
    return np.nansum(np.stack(arrays, axis=0), axis=0)



def _build_flux_map(field, line_names, maps_dir, cache=None, scale=1e-17, deredden=False, intrinsic_ha_hb=INTRINSIC_HA_HB_MAP, R_V=R_V_MAP):
    if deredden:
        return _sum_dereddened_maps(
            field,
            line_names,
            maps_dir=maps_dir,
            cache=cache,
            scale=scale,
            intrinsic_ha_hb=intrinsic_ha_hb,
            R_V=R_V,
        )
    return _sum_maps_from_spec(field, line_names, maps_dir=maps_dir, scale=scale, cache=cache)



def _make_region_panel_map(field, panel_spec, maps_dir, cache=None):
    kind = panel_spec.get('kind', 'ratio').lower()
    use_deredden = panel_spec.get('deredden', False)
    intrinsic_ha_hb = panel_spec.get('intrinsic_ha_hb', INTRINSIC_HA_HB_MAP)
    R_V = panel_spec.get('R_V', R_V_MAP)

    if kind == 'ratio':
        numerator = _build_flux_map(
            field,
            panel_spec['numerator_lines'],
            maps_dir=maps_dir,
            scale=panel_spec.get('numerator_scale', 1e-17),
            cache=cache,
            deredden=use_deredden,
            intrinsic_ha_hb=intrinsic_ha_hb,
            R_V=R_V,
        )
        denominator = _build_flux_map(
            field,
            panel_spec['denominator_lines'],
            maps_dir=maps_dir,
            scale=panel_spec.get('denominator_scale', 1e-17),
            cache=cache,
            deredden=use_deredden,
            intrinsic_ha_hb=intrinsic_ha_hb,
            R_V=R_V,
        )

        min_num = panel_spec.get('min_num')
        min_den = panel_spec.get('min_den')
        if min_num is not None:
            numerator = np.where(numerator >= min_num, numerator, np.nan)
        if min_den is not None:
            denominator = np.where(denominator >= min_den, denominator, np.nan)

        with np.errstate(divide='ignore', invalid='ignore'):
            image = numerator / denominator
        image = np.where(np.isfinite(image), image, np.nan)

        if panel_spec.get('log10', True):
            image = np.where(image > 0, image, np.nan)
            with np.errstate(divide='ignore', invalid='ignore'):
                image = np.log10(image)

    elif kind == 'line':
        image = _build_flux_map(
            field,
            panel_spec['lines'],
            maps_dir=maps_dir,
            scale=panel_spec.get('flux_scale', 1e-17),
            cache=cache,
            deredden=use_deredden,
            intrinsic_ha_hb=intrinsic_ha_hb,
            R_V=R_V,
        )
        min_flux = panel_spec.get('min_flux')
        if min_flux is not None:
            image = np.where(image >= min_flux, image, np.nan)
        if panel_spec.get('log10', False):
            image = np.where(image > 0, image, np.nan)
            with np.errstate(divide='ignore', invalid='ignore'):
                image = np.log10(image)
    else:
        raise ValueError(f"Unsupported panel kind: {kind}")

    finite_clip = panel_spec.get('finite_clip')
    if finite_clip is not None:
        lo, hi = finite_clip
        image = np.where((image >= lo) & (image <= hi), image, np.nan)

    return image



def plot_region_multipanel_maps(
    field,
    region_id,
    panel_specs,
    *,
    maps_dir='../M33-Maps-Calibrated',
    zoi_dir='ZOI_maps/ZOI_map_100pc',
    boundary_dir='Boundary_maps/Boundary_map_100pc',
    boundary_metrics_pattern='Boundary_metrics_{field}.csv',
    zoi_pattern='ZoI_map_{field}.fits',
    boundary_pattern='Boundary_map_{field}.fits',
    ncols=3,
    half_size=None,
    pad_px=5,
    peak_marker_kwargs=None,
    zoi_contour_kwargs=None,
    boundary_contour_kwargs=None,
    show_colorbars=False,
    output_path=None,
    dpi=300,
    suptitle=None,
):
    region_name, region_label = _normalize_region_identifier(field, region_id)

    metrics_path = Path(boundary_dir) / boundary_metrics_pattern.format(field=field)
    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing boundary metrics file: {metrics_path}")
    metrics_df = pd.read_csv(metrics_path)
    reg = metrics_df.loc[metrics_df['region_id'].astype(str) == region_name]
    if len(reg) != 1:
        raise ValueError(f"Could not find unique region {region_name} in {metrics_path}")
    reg = reg.iloc[0]

    cx = float(reg['center_x_px'])
    cy = float(reg['center_y_px'])
    contour_label = int(reg.get('zoi_center_label', region_label))

    zoi_path = Path(zoi_dir) / zoi_pattern.format(field=field)
    boundary_path = Path(boundary_dir) / boundary_pattern.format(field=field)
    if not zoi_path.exists():
        raise FileNotFoundError(f"Missing ZoI map: {zoi_path}")
    if not boundary_path.exists():
        raise FileNotFoundError(f"Missing boundary map: {boundary_path}")

    zoi_map = fits.getdata(zoi_path)
    boundary_map = fits.getdata(boundary_path)

    zoi_mask = np.asarray(zoi_map == contour_label)
    boundary_mask = np.asarray(boundary_map == contour_label)
    if not np.any(zoi_mask):
        raise ValueError(f"Region {region_name} has no ZoI pixels in {zoi_path}")
    if not np.any(boundary_mask):
        raise ValueError(f"Region {region_name} has no boundary pixels in {boundary_path}")

    if half_size is None:
        radius_guess = reg.get('radius_p84_px_after_carve', np.nan)
        if not np.isfinite(radius_guess):
            radius_guess = reg.get('radius_p84_px', np.nan)
        if not np.isfinite(radius_guess):
            radius_guess = reg.get('radius_areaeq_px_after_carve', np.nan)
        if not np.isfinite(radius_guess):
            radius_guess = 35.0
        half_size = int(np.ceil(max(60, 2.25 * float(radius_guess))))

    shape = zoi_mask.shape
    x0 = max(0, int(np.floor(cx - half_size - pad_px)))
    x1 = min(shape[1], int(np.ceil(cx + half_size + pad_px + 1)))
    y0 = max(0, int(np.floor(cy - half_size - pad_px)))
    y1 = min(shape[0], int(np.ceil(cy + half_size + pad_px + 1)))

    peak_x = cx - x0
    peak_y = cy - y0

    if peak_marker_kwargs is None:
        peak_marker_kwargs = dict(marker='+', s=70, linewidths=1.8, color='k')
    if zoi_contour_kwargs is None:
        zoi_contour_kwargs = dict(colors='k', linewidths=1.2, linestyles='--')
    if boundary_contour_kwargs is None:
        boundary_contour_kwargs = dict(colors='black', linewidths=2.5)

    map_cache = {}
    n_panels = len(panel_specs)
    ncols = min(max(1, ncols), n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 5.0 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, spec in zip(axes_flat, panel_specs):
        panel_image = _make_region_panel_map(field, spec, maps_dir=maps_dir, cache=map_cache)
        cutout = panel_image[y0:y1, x0:x1]
        zoi_cutout = zoi_mask[y0:y1, x0:x1]
        boundary_cutout = boundary_mask[y0:y1, x0:x1]

        im = ax.imshow(
            cutout,
            origin='lower',
            cmap=spec.get('cmap', 'viridis'),
            vmin=spec.get('vmin'),
            vmax=spec.get('vmax'),
        )
        # ax.contour(zoi_cutout.astype(float), levels=[0.5], origin='lower', **zoi_contour_kwargs)
        ax.contour(boundary_cutout.astype(float), levels=[0.5], origin='lower', **boundary_contour_kwargs)
        ax.scatter([peak_x], [peak_y], **peak_marker_kwargs)

        # ax.set_title(spec.get('label', ''))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')

        if show_colorbars:
            cbar = fig.colorbar(im, ax=ax, fraction=0.048, pad=0.0, orientation = 'horizontal')
            if spec.get('colorbar_label'):
                cbar.set_label(spec['colorbar_label'], fontsize = 15)

    for ax in axes_flat[n_panels:]:
        ax.axis('off')

    # if suptitle is None:
    #     suptitle = f"{region_name}: line-ratio and line-map cutouts"
    # fig.suptitle(suptitle, y=0.995)
    fig.tight_layout()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight')

    plt.close()
    return fig, axes


def _estimate_region_pixel_scale_pc(reg):
    candidate_pairs = [
        ('radius_areaeq_pc_after_carve', 'radius_areaeq_px_after_carve'),
        ('radius_p84_pc', 'radius_p84_px'),
        ('radius_areaeq_pc', 'radius_areaeq_px'),
        ('radius_p50_pc', 'radius_p50_px'),
        ('radius_p16_pc', 'radius_p16_px'),
    ]
    for pc_col, px_col in candidate_pairs:
        pc_val = reg.get(pc_col, np.nan)
        px_val = reg.get(px_col, np.nan)
        if np.isfinite(pc_val) and np.isfinite(px_val) and float(px_val) > 0:
            return float(pc_val) / float(px_val)
    raise ValueError('Could not infer a pc-per-pixel scale for this region from the boundary metrics table.')


def _make_panel_value_norm(values, panel_spec):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return mpl.colors.Normalize(vmin=0.0, vmax=1.0)

    vmin = panel_spec.get('vmin')
    vmax = panel_spec.get('vmax')
    if vmin is None:
        vmin = float(np.nanpercentile(finite, 5))
    if vmax is None:
        vmax = float(np.nanpercentile(finite, 95))
    if not np.isfinite(vmin):
        vmin = float(np.nanmin(finite))
    if not np.isfinite(vmax):
        vmax = float(np.nanmax(finite))
    if vmax <= vmin:
        vmax = vmin + 1e-6
    return mpl.colors.Normalize(vmin=vmin, vmax=vmax)


def plot_region_multipanel_radial_profiles(
    field,
    region_id,
    panel_specs,
    *,
    maps_dir='../M33-Maps-Calibrated',
    zoi_dir='ZOI_maps/ZOI_map_100pc',
    boundary_dir='Boundary_maps/Boundary_map_100pc',
    boundary_metrics_pattern='Boundary_metrics_{field}.csv',
    zoi_pattern='ZoI_map_{field}.fits',
    boundary_pattern='Boundary_map_{field}.fits',
    ncols=3,
    radial_bin_size_pc=None,
    min_points_per_bin=4,
    scatter_size=14,
    scatter_alpha=0.75,
    median_line_kwargs=None,
    show_colorbars=True,
    output_path=None,
    dpi=300,
    suptitle=None,
):
    region_name, region_label = _normalize_region_identifier(field, region_id)

    metrics_path = Path(boundary_dir) / boundary_metrics_pattern.format(field=field)
    if not metrics_path.exists():
        raise FileNotFoundError(f'Missing boundary metrics file: {metrics_path}')
    metrics_df = pd.read_csv(metrics_path)
    reg = metrics_df.loc[metrics_df['region_id'].astype(str) == region_name]
    if len(reg) != 1:
        raise ValueError(f'Could not find unique region {region_name} in {metrics_path}')
    reg = reg.iloc[0]

    cx = float(reg['center_x_px'])
    cy = float(reg['center_y_px'])
    contour_label = int(reg.get('zoi_center_label', region_label))
    pc_per_px = _estimate_region_pixel_scale_pc(reg)
    equal_area_radius_pc = reg.get('radius_areaeq_pc_after_carve', np.nan)
    if not np.isfinite(equal_area_radius_pc):
        equal_area_radius_pc = reg.get('radius_areaeq_pc', np.nan)

    zoi_path = Path(zoi_dir) / zoi_pattern.format(field=field)
    boundary_path = Path(boundary_dir) / boundary_pattern.format(field=field)
    if not zoi_path.exists():
        raise FileNotFoundError(f'Missing ZoI map: {zoi_path}')
    if not boundary_path.exists():
        raise FileNotFoundError(f'Missing boundary map: {boundary_path}')

    zoi_map = fits.getdata(zoi_path)
    boundary_map = fits.getdata(boundary_path)
    zoi_mask = np.asarray(zoi_map == contour_label)
    boundary_mask = np.asarray(boundary_map == contour_label)
    if not np.any(zoi_mask):
        raise ValueError(f'Region {region_name} has no ZoI pixels in {zoi_path}')
    if not np.any(boundary_mask):
        raise ValueError(f'Region {region_name} has no boundary pixels in {boundary_path}')

    yy, xx = np.indices(boundary_mask.shape, dtype=float)
    radius_pc_map = np.hypot(xx - cx, yy - cy) * pc_per_px

    if median_line_kwargs is None:
        median_line_kwargs = dict(color='k', linewidth=2.0)

    map_cache = {}
    n_panels = len(panel_specs)
    ncols = min(max(1, ncols), n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, spec in zip(axes_flat, panel_specs):
        panel_image = _make_region_panel_map(field, spec, maps_dir=maps_dir, cache=map_cache)
        valid = boundary_mask & np.isfinite(panel_image) & np.isfinite(radius_pc_map)

        if not np.any(valid):
            ax.text(0.5, 0.5, 'No finite values\nin boundary', ha='center', va='center', transform=ax.transAxes)
            ax.set_xlabel('Distance from peak (pc)')
            ax.set_ylabel(spec.get('colorbar_label', 'Value'))
            ax.grid(alpha=0.2, linewidth=0.6)
            continue

        radii_pc = radius_pc_map[valid].astype(float)
        values = panel_image[valid].astype(float)
        order = np.argsort(radii_pc)
        radii_pc = radii_pc[order]
        values = values[order]

        cmap_obj = plt.get_cmap(spec.get('cmap', 'viridis'))
        norm = _make_panel_value_norm(values, spec)
        scat = ax.scatter(
            radii_pc,
            values,
            c=values,
            cmap=cmap_obj,
            norm=norm,
            s=scatter_size,
            alpha=scatter_alpha,
            linewidths=0,
            rasterized=True,
        )

        rmax_pc = float(np.nanmax(radii_pc))
        if radial_bin_size_pc is None:
            n_bins = int(np.clip(np.sqrt(values.size), 12, 40))
            if rmax_pc <= 0:
                bin_edges = np.array([0.0, 1.0])
            else:
                bin_edges = np.linspace(0.0, rmax_pc, n_bins + 1)
        else:
            step_pc = max(float(radial_bin_size_pc), np.finfo(float).eps)
            upper_pc = max(rmax_pc, step_pc)
            bin_edges = np.arange(0.0, upper_pc + step_pc, step_pc)
            if bin_edges.size < 2:
                bin_edges = np.array([0.0, upper_pc])

        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_index = np.clip(np.digitize(radii_pc, bin_edges) - 1, 0, len(bin_centers) - 1)
        medians = np.full(len(bin_centers), np.nan)
        counts = np.zeros(len(bin_centers), dtype=int)
        for idx in range(len(bin_centers)):
            in_bin = values[bin_index == idx]
            counts[idx] = in_bin.size
            if in_bin.size:
                medians[idx] = np.nanmedian(in_bin)
        good_bins = (counts >= int(min_points_per_bin)) & np.isfinite(medians)
        if np.any(good_bins):
            ax.plot(bin_centers[good_bins], medians[good_bins], **median_line_kwargs)

        x_upper = max(float(bin_edges[-1]), rmax_pc)
        if np.isfinite(equal_area_radius_pc):
            x_upper = max(x_upper, float(equal_area_radius_pc))
        ax.set_xlim(0.0, x_upper)
        y_lower = float(np.nanmin(values))
        y_upper = float(np.nanmax(values))
        if spec.get('vmin') is not None:
            y_lower = min(y_lower, float(spec['vmin']))
        if spec.get('vmax') is not None:
            y_upper = max(y_upper, float(spec['vmax']))
        if y_upper <= y_lower:
            y_pad = max(1e-6, 0.05 * max(abs(y_lower), 1.0))
        else:
            y_pad = 0.05 * (y_upper - y_lower)
        ax.set_ylim(y_lower - y_pad, y_upper + y_pad)
        if np.isfinite(equal_area_radius_pc):
            ax.axvline(float(equal_area_radius_pc), color='0.25', linestyle='--', linewidth=1.5)
        ax.set_xlabel('Distance from peak (pc)')
        ax.set_ylabel(spec.get('colorbar_label', 'Value'))
        ax.grid(alpha=0.2, linewidth=0.6)

        if show_colorbars:
            cbar = fig.colorbar(scat, ax=ax, fraction=0.048, pad=0.0, orientation='horizontal')
            if spec.get('colorbar_label'):
                cbar.set_label(spec['colorbar_label'], fontsize=15)

    for ax in axes_flat[n_panels:]:
        ax.axis('off')

    if suptitle is not None:
        fig.suptitle(suptitle, y=0.995)
    fig.tight_layout()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=dpi, bbox_inches='tight')

    plt.close()
    return fig, axes

dered = True
DEFAULT_REGION_RATIO_PANEL_SPECS = [
    dict(
        kind='ratio',
        numerator_lines='OIII5007flux',
        denominator_lines='OII3727flux',
        colorbar_label=r'log([O III]/[O II])',
        cmap='BrBG_r',
        log10=True,
        deredden=dered,
        vmin=-1,
        vmax=0.5,
        min_den=1e-3,
    ),
    dict(
        kind='ratio',
        numerator_lines='OIII5007flux',
        denominator_lines=['SII6716flux', 'SII6731flux'],
        colorbar_label=r'log([O III]/[S II])',
        cmap='PiYG_r',
        log10=True,
        deredden=dered,
        vmin=-1,
        vmax=1,
        min_den=1e-3,
    ),
    dict(
        kind='ratio',
        numerator_lines='Haflux',
        denominator_lines='Hbflux',
        colorbar_label=r'H$\alpha$/H$\beta$',
        cmap='coolwarm',
        log10=False,
        deredden=False,
        vmin=2,
        vmax=5,
        min_den=1e-3,
    ),
    dict(
        kind='ratio',
        numerator_lines='OIII5007flux',
        denominator_lines='Hbflux',
        colorbar_label=r'log([O III]/H$\beta$)',
        cmap='bwr_r',
        log10=True,
        deredden=dered,
        vmin=-1,
        vmax=1,
        min_den=1e-3,
    ),
    dict(
        kind='ratio',
        numerator_lines=['SII6716flux', 'SII6731flux'],
        denominator_lines='Haflux',
        colorbar_label=r'log([S II]/H$\alpha$)',
        cmap='PuOr',
        log10=True,
        deredden=dered,
        vmin=-1.5,
        vmax=0.8,
        min_den=1e-3,
    ),
    
    dict(
        kind='ratio',
        numerator_lines='NII6584flux',
        denominator_lines='Haflux',
        colorbar_label=r'log(N II]/H$\alpha$)',
        cmap='PRGn',
        log10=True,
        deredden=dered,
        vmin=-1.5,
        vmax=0,
        min_den=1e-3,
    ),
    
    dict(
        kind='ratio',
        numerator_lines='SII6716flux',
        denominator_lines='SII6731flux',
        colorbar_label=r'[S II] 6716/[S II] 6731',
        cmap='plasma',
        log10=False,
        deredden=dered,
        vmin=1,
        vmax=1.45,
        min_den=1e-3,
    ),
]

DEFAULT_REGION_LINE_PANEL_SPECS = [
    # dict(kind='line', lines='OIII5007flux', label=r'log([O III] 5007 flux), dered.', cmap='rainbow', log10=True, deredden=True, vmin=-1, vmax=2, min_flux=1e-3),
    dict(kind='line', lines='Haflux', colorbar_label=r'log(H$\alpha$/$10^{-17}$erg s$^{-1}$cm$^{-2}$)', cmap='rainbow', log10=True, vmin=-1, vmax=2, deredden=False),
    # dict(kind='line', lines='Hbflux', label=r'log(H$\beta$ flux), dered.', cmap='rainbow', log10=True, deredden=True, vmin=-1, vmax=2, min_flux=1e-3),
    # dict(kind='line', lines='NII6584flux', label=r'log([N II] 6584 flux), dered.', cmap='rainbow', log10=True, deredden=True, vmin=-1, vmax=2, min_flux=1e-3),
    # dict(kind='line', lines='OII3727flux', label=r'log([O II] 3727 flux), dered.', cmap='rainbow', log10=True, deredden=True, vmin=-1, vmax=2, min_flux=1e-3),
    # dict(kind='line', lines=['SII6716flux', 'SII6731flux'], label=r'log([S II] 6716, 6731 flux), dered.', cmap='rainbow', log10=True, deredden=True, vmin=-1, vmax=2, min_flux=1e-3),
]


In [ ]:
# Example figure: region NW_0824, matching the notebook 5 example region.
#
# By default these panels use Balmer-decrement dereddened flux maps for all panels
# except the observed Halpha/Hbeta panel.
# To add more panels, append specs from DEFAULT_REGION_LINE_PANEL_SPECS or add your own dict.
# Example:
region_panel_specs = list(DEFAULT_REGION_LINE_PANEL_SPECS)+ list(DEFAULT_REGION_RATIO_PANEL_SPECS)
# region_panel_specs.append(dict(
#     kind='ratio',
#     numerator_lines='NII6584flux',
#     denominator_lines='Haflux',
#     label=r'log([N II] 6584 / H$\alpha$), dered.',
#     cmap='coolwarm',
#     log10=True,
#     deredden=True,
#     vmin=-2,
#     vmax=0,
#     min_den=1e-3,
# ))

# region_panel_specs = list(DEFAULT_REGION_RATIO_PANEL_SPECS)

# ids = [100,200,300,400,500,600, 824]
ids = [850]
# ids = np.arange(840, 860)

for id in ids:
    # try:
    id = str(id)
    while len(id)<4:
        id='0'+id
    
    fig_profile, axes_profile = plot_region_multipanel_radial_profiles(
        field='NW',
        region_id=f'NW_{id}',
        panel_specs=region_panel_specs,
        ncols=4,
        output_path=str(paper_plot_path(f'NW_0{id}_region_ratio_radial_profiles_dered.png', subdir='region_profiles')),
        suptitle=None,
        show_colorbars=True,
    )
    print(id, ' sucess')
    # except:
    #     print(id, 'fail')


In [ ]:
str(paper_plot_path(f'NW_0{id}_region_ratio_radial_profiles_dered.png', subdir='region_profiles')),

In [ ]:
# Luminosity function for the star-forming, high-SNR subset.
if 'luminosity_function_pdf_points' not in globals():
    def luminosity_function_pdf_points(logL, bins):
        logL = np.asarray(logL, dtype=float)
        logL = logL[np.isfinite(logL)]
        if len(logL) == 0:
            return np.array([]), np.array([]), np.array([])
        counts, edges = np.histogram(logL, bins=bins)
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = np.diff(edges)
        pdf = counts / (np.sum(counts) * widths)
        detected = counts > 0
        return centers[detected], pdf[detected], counts[detected]
if 'logL_bins' not in globals():
    logL_bins = np.arange(33.0, 40.6, 0.2)
sf_logL = pd.to_numeric(cat_starforming_snrthree['log_L_Ha_sum_dered'], errors='coerce').to_numpy(dtype=float)
sf_logL = sf_logL[np.isfinite(sf_logL)]
fig, ax = plt.subplots(figsize=(8, 6))
centers, pdf, counts = luminosity_function_pdf_points(sf_logL, logL_bins)
if len(centers) > 0:
    ax.plot(
        centers,
        pdf,
        marker='o',
        markersize=6,
        linewidth=2.5,
        color=color_dic['L_Ha_sum_dered'],
        label=f'Selected SF, S/N>3 (N={len(sf_logL)})',
    )
ax.set_xlabel(r'log$_{10}$[$L_{\rm H\alpha}$ (erg s$^{-1}$)]')
ax.set_ylabel(r'Probability density per dex')
ax.set_yscale('log')
ax.minorticks_on()
ax.legend(frameon=False, fontsize=12)
fig.tight_layout()
fig.savefig(
    sf_plot_path('M33_HII_Halpha_luminosity_function_pdf.png', subdir='luminosity_functions'),
    dpi=300,
    bbox_inches='tight',
)
plt.close(fig)

sf_L = pd.to_numeric(cat_starforming_snrthree['L_Ha_sum_dered'], errors='coerce').to_numpy(dtype=float)
# Santoro+2022 / Clauset+2009 LF fit for the star-forming sample:
# test every empirical Lmin in median ±1 sigma, restrict alpha to 1-3,
# and bootstrap Lmin uncertainty with 1000 mock LFs.
sf_luminosity_function_results = santoro_powerlaw_fit_fast(
    sf_L,
    alpha_range=(1.0, 3.0),
    xmin_stride=1,
    min_tail_n=30,
    n_bootstrap=1000,
    verbose=True,
)
sf_sigma_alpha, sf_alpha_grid, sf_rel_like, sf_gfit = alpha_uncertainty_from_likelihood(
    sf_luminosity_function_results['L'],
    sf_luminosity_function_results['xmin_best'],
    sf_luminosity_function_results['alpha_best'],
    alpha_range=(1.0, 3.0),
    n_alpha_grid=400,
)
print(f'Star-forming SNR>3 alpha uncertainty ~ {sf_sigma_alpha:.3f}')

_sigma_alpha_full_catalog = sigma_alpha
try:
    sigma_alpha = sf_sigma_alpha
    plot_histogram_with_unbinned_powerlaw_fit(
        sf_luminosity_function_results,
        bins=100,
        outfile=str(sf_plot_path(
            'M33_HII_hist_with_fast_unbinned_powerlaw_fit.png',
            subdir='luminosity_function',
        )),
    )
finally:
    sigma_alpha = _sigma_alpha_full_catalog

# Rewrite the Overleaf numbers so unsuffixed fit commands refer to the
# star-forming SNR>3 sample, and all-catalog fit commands carry an All suffix.
def add_fit_values_with_suffix(values, formats, fit, prefix, suffix='All', fmt='.4f'):
    if not fit:
        return
    keys = {
        'slope': f'{prefix}Slope{suffix}',
        'intercept': f'{prefix}Intercept{suffix}',
        'slope_stderr': f'{prefix}SlopeErr{suffix}',
        'intercept_stderr': f'{prefix}InterceptErr{suffix}',
        'n_points': f'{prefix}N{suffix}',
    }
    for source, command in keys.items():
        if source not in fit:
            continue
        values[command] = fit[source]
        if source != 'n_points':
            formats[command] = fmt

for fit_name, fit_results in all_catalog_radial_gradient_fit_results.items():
    add_fit_values_with_suffix(values, formats, fit_results.get('point_fit_stats'), fit_name, suffix='All')
    add_fit_values_with_suffix(values, formats, fit_results.get('binned_fit_stats'), fit_name, suffix='BinsAll')

def clear_fit_values(values, formats, prefix):
    for suffix in ['Slope', 'Intercept', 'SlopeErr', 'InterceptErr', 'N']:
        values.pop(f'{prefix}{suffix}', None)
        formats.pop(f'{prefix}{suffix}', None)

for fit_name in all_catalog_radial_gradient_fit_results:
    clear_fit_values(values, formats, fit_name)
    clear_fit_values(values, formats, f'{fit_name}Bins')

for fit_name, fit_results in radial_gradient_fit_results.items():
    add_fit_values(values, formats, fit_results.get('point_fit_stats'), fit_name)
    add_fit_values(values, formats, fit_results.get('binned_fit_stats'), f'{fit_name}Bins')

# Preserve full-catalog luminosity-function values with an All suffix.
values['luminosityFunctionAlphaAll'] = luminosity_function_results['alpha_best']
values['luminosityFunctionAlphaErrAll'] = sigma_alpha
values['luminosityFunctionSlopeAll'] = -luminosity_function_results['alpha_best']
values['luminosityFunctionSlopeErrAll'] = sigma_alpha
values['luminosityFunctionLogLminAll'] = np.log10(luminosity_function_results['xmin_best'])
values['luminosityFunctionLogLminErrAll'] = luminosity_function_results['xmin_err'] / (luminosity_function_results['xmin_best'] * np.log(10.0))
values['luminosityFunctionKSDistanceAll'] = luminosity_function_results['ks_best']
values['luminosityFunctionNTailAll'] = luminosity_function_results['n_tail_best']

# Use the current unsuffixed luminosity-function names for the star-forming SNR>3 sample.
values['luminosityFunctionAlpha'] = sf_luminosity_function_results['alpha_best']
values['luminosityFunctionAlphaErr'] = sf_sigma_alpha
values['luminosityFunctionSlope'] = -sf_luminosity_function_results['alpha_best']
values['luminosityFunctionSlopeErr'] = sf_sigma_alpha
values['luminosityFunctionLogLmin'] = np.log10(sf_luminosity_function_results['xmin_best'])
values['luminosityFunctionLogLminErr'] = sf_luminosity_function_results['xmin_err'] / (sf_luminosity_function_results['xmin_best'] * np.log(10.0))
values['luminosityFunctionKSDistance'] = sf_luminosity_function_results['ks_best']
values['luminosityFunctionNTail'] = sf_luminosity_function_results['n_tail_best']

formats.update({
    'luminosityFunctionAlpha': '.4f',
    'luminosityFunctionAlphaErr': '.4f',
    'luminosityFunctionSlope': '.4f',
    'luminosityFunctionSlopeErr': '.4f',
    'luminosityFunctionLogLmin': '.3f',
    'luminosityFunctionLogLminErr': '.3f',
    'luminosityFunctionKSDistance': '.4f',
    'luminosityFunctionAlphaAll': '.4f',
    'luminosityFunctionAlphaErrAll': '.4f',
    'luminosityFunctionSlopeAll': '.4f',
    'luminosityFunctionSlopeErrAll': '.4f',
    'luminosityFunctionLogLminAll': '.3f',
    'luminosityFunctionLogLminErrAll': '.3f',
    'luminosityFunctionKSDistanceAll': '.4f',
})

add_halpha_flux_partition_values(values, formats)
catalog_numbers_path = paper_plot_path('catalog_numbers.tex')
write_latex_commands(catalog_numbers_path, values, formats)
print(
    f'Rewrote {catalog_numbers_path} with unsuffixed fit commands from the '
    f'star-forming SNR>3 sample and All-suffixed commands from the full catalog.'
)


# Luminosity versus electron density for the star-forming, high-SNR subset.
sf_lha_ne_mask = (
    np.isfinite(cat_starforming_snrthree['L_Ha_sum_dered']) &
    np.isfinite(cat_starforming_snrthree['ne_SII_cm3']) &
    cat_starforming_snrthree['ne_SII_reliable'].fillna(False) &
    (cat_starforming_snrthree['ne_SII_cm3'] > 10) &
    (cat_starforming_snrthree['ne_SII_cm3'] < 10000) &
    (cat_starforming_snrthree['L_Ha_sum_dered'] > 1e34) &
    (cat_starforming_snrthree['L_Ha_sum_dered'] < 1e40)
)
plot_radial_gradient(
    cat=cat_starforming_snrthree,
    xcol='ne_SII_cm3',
    ycol='L_Ha_sum_dered',
    mask=sf_lha_ne_mask,
    bins=np.logspace(1, 4, 16),
    color=color_dic['ne_SII_cm3'],
    xlabel=r'$n_e$ (cm$^{-3}$ )',
    ylabel=r'$L_{\rm H\alpha}$ (erg s$^{-1}$)',
    xscale='log',
    yscale='log',
    fit_bins=False,
    alpha=0.2,
    fit_logx_points=True,
    fit_logy_points=True,
    fit_equation_x=r'n_e',
    fit_equation_y=r'L_{\rm H\alpha}',
    fit_name='scalingNeLHa',
    savepath=str(sf_plot_path('M33_HII_L_Ha_sum_dered_vs_ne_SII_cm3.png', subdir='scaling_relations')),
    show=False,
)


In [ ]:
# Diagnostic LF split at log L_Halpha = 38.6 for final corrected star-forming regions.
# The high-luminosity side has very few regions in the current catalog, so the fit is guarded.

lf_split_logL = 38.6
lf_split_L = 10.0 ** lf_split_logL

if 'cat_digsub_sf_lf' not in globals():
    cat_digsub_sf_lf = _starforming_lf_matched_catalog(cat_digsub)

final_sf_L = _positive_luminosities(cat_digsub_sf_lf, 'L_Ha_sum_dered')
lf_split_samples = [
    dict(
        label=rf'$L_{{\rm H\alpha}} \leq 10^{{{lf_split_logL:.1f}}}$',
        L=final_sf_L[final_sf_L <= lf_split_L],
        color=color_dic['L_Ha_sum_dered'],
        marker='o',
    ),
    dict(
        label=rf'$L_{{\rm H\alpha}} > 10^{{{lf_split_logL:.1f}}}$',
        L=final_sf_L[final_sf_L > lf_split_L],
        color='forestgreen',
        marker='s',
    ),
]

fig, ax = plt.subplots(figsize=(8.8, 6.6))
lf_split_fit_results = {}
min_tail_n_split = 30

for sample in lf_split_samples:
    L_values = sample['L']
    logL_values = np.log10(L_values[np.isfinite(L_values) & (L_values > 0)])
    centers, pdf, counts = luminosity_function_pdf_points(logL_values, logL_bins)
    if len(centers) == 0:
        ax.plot([], [], color=sample['color'], marker=sample['marker'], label=f"{sample['label']} (N=0)")
        continue

    fit_results = None
    fit_label = f"{sample['label']} (N={len(logL_values)})"
    if len(L_values) >= min_tail_n_split:
        try:
            fit_results = santoro_powerlaw_fit_fast(
                L_values,
                alpha_range=(1.0, 3.0),
                xmin_stride=1,
                min_tail_n=min_tail_n_split,
                n_bootstrap=1000,
                verbose=False,
            )
            sigma_alpha, alpha_grid, rel_like, gfit = alpha_uncertainty_from_likelihood(
                fit_results['L'],
                fit_results['xmin_best'],
                fit_results['alpha_best'],
                alpha_range=(1.0, 3.0),
                n_alpha_grid=400,
            )
            fit_results['alpha_err'] = sigma_alpha
            lf_split_fit_results[sample['label']] = fit_results
            fit_label += rf"; $\alpha={fit_results['alpha_best']:.2f}\pm{sigma_alpha:.2f}$, slope$={-fit_results['alpha_best']:.2f}$"
        except ValueError as exc:
            fit_label += f"; Santoro fit unavailable ({exc})"
    else:
        fit_label += rf"; Santoro fit unavailable ($N<{min_tail_n_split}$)"

    ax.plot(
        centers,
        pdf,
        marker=sample['marker'],
        markersize=6,
        linewidth=2.2,
        color=sample['color'],
        label=fit_label,
    )

    if fit_results is not None:
        log_lmin = np.log10(fit_results['xmin_best'])
        fit_logL = np.linspace(log_lmin, max(np.nanmax(centers), log_lmin + 0.2), 200)
        ax.plot(
            fit_logL,
            _powerlaw_pdf_per_dex(fit_logL, fit_results),
            color=sample['color'],
            linestyle='--',
            linewidth=1.8,
            alpha=0.9,
        )
        ax.axvline(
            log_lmin,
            color=sample['color'],
            linestyle=':',
            linewidth=1.8,
            alpha=0.9,
        )

ax.axvline(lf_split_logL, color='0.25', linestyle='-', linewidth=1.4, alpha=0.8, label=rf'Split: $\log L_{{\rm H\alpha}}={lf_split_logL:.1f}$')
ax.set_xlabel(r'log$_{10}$[$L_{\rm H\alpha}$ (erg s$^{-1}$)]')
ax.set_ylabel('Empirical PDF per dex')
ax.set_yscale('log')
ax.minorticks_on()
ax.legend(frameon=False, fontsize=9.2, loc='best')
fig.tight_layout()
fig.savefig(
    sf_plot_path('M33_HII_Halpha_luminosity_function_final_corrected_split_logL38p6_empirical_pdf.png', subdir='luminosity_functions'),
    dpi=300,
    bbox_inches='tight',
)
plt.close(fig)

for sample in lf_split_samples:
    print(f"{sample['label']}: N={len(sample['L'])}")
